# StaticKD: Distilasi Pengetahuan XLM-R ke Model *Static-Embedding* untuk Klasifikasi Topik Berita IPTC Multibahasa yang Ringan

**Notebook riset — versi Google Colab**

Notebook ini adalah implementasi lengkap dan terdokumentasi dari riset klasifikasi artikel berita ke **17 label top-level IPTC Media Topic** dengan target **real-time di CPU** (tanpa GPU), secepat fastText *language identification*, tetapi tetap multibahasa dan akurat.

| | |
|---|---|
| **Model acuan (teacher)** | `classla/multilingual-IPTC-news-topic-classifier` (Kuzman & Ljubešić, 2025, *IEEE Access*) — XLM-RoBERTa-large, 560 juta parameter |
| **Masalah** | Akurat, tetapi di CPU butuh ~1–2,6 detik per artikel dan RAM 1,4–3 GB → tidak layak untuk pipeline real-time di server kecil |
| **Metode usulan** | **StaticKD** — teacher melabeli korpus berita multibahasa besar (70 bahasa) → student berupa *static multilingual embeddings* + *mean pooling* + lapisan linear, dilatih dengan *knowledge distillation* (KL-divergence ke distribusi teacher) → dilipat menjadi tabel 17 logit per token |
| **Hasil utama** | ~720× lebih cepat dari teacher ONNX-int8 di 1 core CPU, file 34 MB, mempertahankan ~91% macro-F1 teacher di 4 bahasa latih dan 16 bahasa lain |

---

### Pertanyaan penelitian
- **RQ1** — Seberapa akurat dan seberapa mahal (CPU) model IPTC Kuzman & Ljubešić bila dijalankan di server tanpa GPU?
- **RQ2** — Dapatkah pengetahuan teacher didistilasi ke model *static-embedding* secepat fastText tanpa kehilangan kemampuan multibahasa?
- **RQ3** — Komponen apa yang paling menentukan keberhasilan distilasi (sumber label, cakupan bahasa data, inisialisasi embedding, regularisasi, ukuran data)?
- **RQ4** — Bagaimana trade-off akurasi–efisiensi dibanding baseline fastText dan student transformer kecil, dan apakah ada bahasa/label yang sistematis lebih lemah?

### Struktur notebook
| Bagian | Isi |
|---|---|
| 0 | Konfigurasi, mode eksekusi, instalasi |
| 1 | Modul pustaka (konfigurasi, metrik statistik, gaya visualisasi) |
| 2 | Data: EMMediaTopic, korpus CC-News multibahasa, test set LLM 16 bahasa |
| 3 | Review model teacher (Kuzman & Ljubešić): akurasi, statistik, validasi anotasi |
| 4 | Pelabelan distilasi (*soft label*) oleh teacher |
| 5 | Model student: fastText, **StaticKD**, ablasi, transformer kecil, kompresi |
| 6 | Evaluasi akurasi terpadu + uji signifikansi |
| 7 | Efisiensi CPU (latensi, throughput, RAM) |
| 8 | Analisis: per bahasa, per label, kalibrasi, cascade, skala data, contoh kualitatif |
| 9 | Ekspor tabel & gambar untuk paper, ringkasan temuan |
| 10 | Keterbatasan & catatan reproduktibilitas |

## 0. Cara menggunakan notebook ini

Notebook memiliki **dua mode**:

| Mode | Apa yang dijalankan | Kebutuhan | Waktu |
|---|---|---|---|
| **`REPORT`** (default) | Semua **analisis, statistik, tabel, dan gambar** dibuat ulang dari hasil eksperimen yang sudah ada (bundle `iptc_statickd_bundle.zip`). Tidak ada training/pelabelan ulang. | CPU saja | ± 5–10 menit |
| **`FULL`** | Seluruh pipeline dari nol: unduh data → pelabelan teacher → training semua student → evaluasi → benchmark CPU → analisis. | GPU (T4 cukup), ± 15 GB disk Drive | ± 6–10 jam (bisa dilanjutkan; setiap tahap menyimpan checkpoint ke Drive) |

**Langkah untuk mode `REPORT`:**
1. Unggah `iptc_statickd_bundle.zip` ke Google Drive pada folder `MyDrive/iptc_statickd/`.
2. Jalankan semua sel (*Runtime → Run all*). Bundle akan diekstrak otomatis.

**Langkah untuk mode `FULL`:**
1. Pilih *Runtime → Change runtime type → T4 GPU*.
2. Ubah `MODE = "FULL"` pada sel konfigurasi di bawah.
3. (Opsional) Isi `ANTHROPIC_API_KEY` di *Colab Secrets* jika ingin menganotasi ulang test set 16 bahasa dengan Claude. Tanpa itu, unggah `llm_test.jsonl` (ada di bundle).
4. Jalankan semua sel. Jika sesi Colab terputus, jalankan ulang: tahap yang sudah selesai akan dilewati karena hasilnya sudah tersimpan di Drive.

> **Tentang `SMOKE_TEST`**: bila `True`, setiap tahap FULL dijalankan dengan data sangat kecil (beberapa ratus dokumen, 1 epoch) untuk memastikan kode berjalan dari awal sampai akhir dalam ±15 menit. Angka hasilnya tidak bermakna secara ilmiah.

> **Catatan hardware**: angka efisiensi pada mode REPORT diukur di laptop Intel Core Ultra 7 155H (proses dipin ke P-core). Pada mode FULL, benchmark diulang di CPU Colab (umumnya 2 vCPU Xeon) dan disimpan terpisah (`cpu_benchmark_colab.jsonl`), sehingga kedua set angka tidak tercampur.

In [ ]:
# ---------------------------------------------------------------------------------------------
# 0.1 Konfigurasi eksekusi — ubah nilai di sini
# ---------------------------------------------------------------------------------------------
import os, sys, json, time, platform, subprocess, zipfile, shutil, warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

MODE = os.environ.get("IPTC_MODE", "REPORT")          # "REPORT" atau "FULL"
SMOKE_TEST = os.environ.get("IPTC_SMOKE_TEST", "0") == "1"   # True = uji cepat dengan data kecil
USE_DRIVE = True                                       # simpan data/model/hasil di Google Drive
BUNDLE_NAME = "iptc_statickd_bundle.zip"               # hasil eksperimen untuk mode REPORT

# Tahap opsional di mode FULL (semuanya True = replikasi penuh paper)
RUN_ABLATIONS = True             # varian student untuk tabel ablasi
RUN_TRANSFORMER_STUDENT = True   # student MiniLM-L6 (butuh ±1 jam GPU)
RUN_DATA_SCALING = True          # 156 ribu dokumen tambahan (butuh ±1 jam GPU untuk pelabelan)
ANNOTATE_WITH_CLAUDE = False     # anotasi ulang test set 16 bahasa via Claude API (berbayar)
RUN_TEACHER_DEMO = False         # prediksi teacher pada contoh kualitatif (unduh model 2,2 GB)

# Anchor every path to the directory of the first run, so re-running this cell is safe.
START_DIR = os.environ.setdefault("IPTC_START_DIR", os.getcwd())
if IN_COLAB and USE_DRIVE and "IPTC_BASE_DIR" not in os.environ:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/iptc_statickd"
else:
    BASE_DIR = os.environ.get("IPTC_BASE_DIR", os.path.join(START_DIR, "iptc_statickd"))

os.environ["IPTC_BASE_DIR"] = BASE_DIR
os.environ["IPTC_SMOKE_TEST"] = "1" if SMOKE_TEST else "0"
os.makedirs(BASE_DIR, exist_ok=True)

# The notebook writes its code as a small importable package (iptc/) so that the CPU benchmark
# can run each model in a fresh subprocess.
LIB_DIR = os.path.join(START_DIR, "iptc_lib")
os.makedirs(os.path.join(LIB_DIR, "iptc"), exist_ok=True)
os.chdir(LIB_DIR)
if LIB_DIR not in sys.path:
    sys.path.insert(0, LIB_DIR)

print(f"Colab: {IN_COLAB} | MODE: {MODE} | SMOKE_TEST: {SMOKE_TEST}")
print(f"BASE_DIR (data, model, hasil): {BASE_DIR}")
print(f"LIB_DIR  (kode notebook)     : {LIB_DIR}")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 0.2 Instalasi paket (hanya di Colab; lingkungan lokal diasumsikan sudah lengkap)
# ---------------------------------------------------------------------------------------------
def pip_install(*packages: str) -> bool:
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-1500:])
    return result.returncode == 0

if IN_COLAB:
    pip_install("onnx", "onnxruntime", "onnxscript", "anthropic", "tabulate")
    # fastText: prebuilt wheel first, source build as fallback.
    if not pip_install("fasttext-wheel"):
        pip_install("fasttext")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy, sklearn, tokenizers

print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", scipy.__version__,
      "| sklearn", sklearn.__version__, "| tokenizers", tokenizers.__version__)
try:
    import torch
    print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
except ImportError:
    print("torch tidak tersedia (hanya dibutuhkan pada mode FULL)")
if MODE == "FULL" and not SMOKE_TEST:
    assert "torch" in sys.modules and torch.cuda.is_available(), \
        "Mode FULL membutuhkan GPU: Runtime -> Change runtime type -> T4 GPU"

In [ ]:
# ---------------------------------------------------------------------------------------------
# 0.3 Mode REPORT: ekstrak bundle hasil eksperimen
# ---------------------------------------------------------------------------------------------
bundle_path = Path(BASE_DIR) / BUNDLE_NAME
marker = Path(BASE_DIR) / "results" / "accuracy"
if MODE == "REPORT":
    if marker.exists() and any(marker.glob("*.json")):
        print("Hasil eksperimen sudah tersedia di", marker.parent)
    elif bundle_path.exists():
        with zipfile.ZipFile(bundle_path) as zf:
            zf.extractall(BASE_DIR)
        print(f"Bundle diekstrak: {len(zf.namelist())} file -> {BASE_DIR}")
    else:
        raise FileNotFoundError(
            f"Mode REPORT membutuhkan {bundle_path}. Unggah {BUNDLE_NAME} ke folder tersebut, "
            "atau ubah MODE = 'FULL'.")

---
## 1. Modul pustaka

Seluruh kode riset ditulis sebagai paket Python kecil `iptc/` menggunakan `%%writefile`. Alasannya:
1. Kode tetap terbaca di notebook (setiap modul dijelaskan sebelum dipakai).
2. Benchmark CPU menjalankan setiap model di **proses baru**, agar pengukuran RAM bersih dari sisa memori model lain. Proses baru hanya bisa memakai kode yang ada di file.

> Jika Anda mengubah isi sel `%%writefile` setelah modul diimpor, jalankan `importlib.reload(modul)` atau restart runtime.

### 1.1 Konfigurasi bersama
Label, jalur direktori, dan praproses mengikuti Kuzman & Ljubešić (2025): input berupa **teks artikel (judul + isi), dipotong ke 512 kata pertama**, dan model card merekomendasikan panjang minimal 75 kata.

In [ ]:
%%writefile iptc/__init__.py
"""IPTC StaticKD research package (written by the notebook)."""

In [ ]:
%%writefile iptc/config.py
"""Paths, labels and constants shared by every module."""
import os
from pathlib import Path

BASE_DIR = Path(os.environ.get("IPTC_BASE_DIR", "iptc_statickd")).resolve()
SMOKE_TEST = os.environ.get("IPTC_SMOKE_TEST", "0") == "1"

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
TEACHER_DIR = PROCESSED_DIR / "teacher_labels"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"
ACCURACY_DIR = RESULTS_DIR / "accuracy"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "paper_tables"
for _d in (RAW_DIR, PROCESSED_DIR, TEACHER_DIR, MODELS_DIR, ACCURACY_DIR, FIGURES_DIR, TABLES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

EMMEDIATOPIC_URL = ("https://www.clarin.si/repository/xmlui/bitstream/handle/11356/1991/"
                    "EMMediaTopic-1.0.jsonl?sequence=1&isAllowed=y")
EMMEDIATOPIC_PATH = RAW_DIR / "EMMediaTopic-1.0.jsonl"
EMM_DEV_PATH = PROCESSED_DIR / "emm_dev.jsonl"          # dev split only (shipped in the bundle)
LLM_TEST_PATH = PROCESSED_DIR / "llm_test.jsonl"
DATA_STATS_PATH = RESULTS_DIR / "data_stats.json"
TRAINING_HISTORY_PATH = RESULTS_DIR / "training_history.json"

TEACHER_MODEL_ID = "classla/multilingual-IPTC-news-topic-classifier"
TEACHER_ONNX_ID = "onnx-community/multilingual-IPTC-news-topic-classifier-ONNX"

# Same preprocessing as Kuzman & Ljubesic (2025): first 512 words; model card: >= 75 words.
MAX_WORDS = 512
MIN_WORDS = 75

# The 17 top-level IPTC Media Topic labels, in the fixed order used by every student.
LABELS = [
    "arts, culture, entertainment and media",
    "conflict, war and peace",
    "crime, law and justice",
    "disaster, accident and emergency incident",
    "economy, business and finance",
    "education",
    "environment",
    "health",
    "human interest",
    "labour",
    "lifestyle and leisure",
    "politics",
    "religion",
    "science and technology",
    "society",
    "sport",
    "weather",
]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
SHORT_LABELS = ["arts/culture", "conflict", "crime/law", "disaster", "economy", "education",
                "environment", "health", "human interest", "labour", "lifestyle", "politics",
                "religion", "sci/tech", "society", "sport", "weather"]

# Languages of the teacher's fine-tuning data (EMMediaTopic).
TEACHER_TRAIN_LANGS = ["ca", "el", "hr", "sl"]


def truncate_words(text: str, max_words: int = MAX_WORDS) -> str:
    """Keep the first `max_words` whitespace-separated words, as in the original paper."""
    return " ".join(text.split()[:max_words])

### 1.2 Metrik dan uji statistik

Semua metrik dihitung dengan definisi yang sama dengan repositori benchmark Kuzman (`sklearn.metrics.f1_score`):

- **Macro-F1** — rata-rata F1 per label (setiap label berbobot sama), metrik utama di paper Kuzman karena menilai juga label yang jarang.
- **Micro-F1** — untuk klasifikasi *single-label* sama dengan akurasi.
- **Interval kepercayaan 95%** — *bootstrap percentile* (resampling dokumen dengan pengembalian, $B = 1000$).
- **Uji beda dua model pada data yang sama** (berpasangan):
  - *Paired bootstrap* untuk selisih macro-F1: $\Delta = F1_A - F1_B$, p-value dua sisi $= 2\cdot\min(P(\Delta^* \le 0), P(\Delta^* \ge 0))$.
  - *McNemar exact test* untuk akurasi: memakai jumlah dokumen yang benar di A tapi salah di B ($b$) dan sebaliknya ($c$), $p$ dari uji binomial $b \sim \text{Bin}(b+c, 0.5)$.
  - Koreksi perbandingan berganda **Holm–Bonferroni**.
- **Kesepakatan anotator** — Cohen's $\kappa$ dan Krippendorff's $\alpha$ (nominal), sama dengan metrik yang dipakai Kuzman untuk membandingkan GPT-4o dengan anotator manusia.
- **Kalibrasi** — *Expected Calibration Error*: $\text{ECE} = \sum_{m=1}^{M} \frac{|B_m|}{n}\,|\text{acc}(B_m) - \text{conf}(B_m)|$ dengan $M = 15$ bin confidence.

In [ ]:
%%writefile iptc/metrics.py
"""Classification metrics and the statistical tests used in the paper."""
import numpy as np
from scipy import stats
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, f1_score

from iptc.config import LABELS, NUM_LABELS


def f1_scores(y_true, y_pred) -> dict:
    """Macro/micro F1 exactly as in Kuzman's benchmark code."""
    present = sorted(set(np.asarray(y_true).tolist()))
    return {
        "macro_f1": float(f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(y_true, y_pred, labels=list(range(NUM_LABELS)), average="micro",
                                   zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "n": int(len(y_true)),
    }


def fast_macro_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Vectorised macro-F1 over the labels present in y_true (used inside bootstrap loops)."""
    cm = np.bincount(y_true * NUM_LABELS + y_pred, minlength=NUM_LABELS ** 2).reshape(NUM_LABELS, NUM_LABELS)
    tp = np.diag(cm).astype(float)
    support, predicted = cm.sum(1), cm.sum(0)
    denom = support + predicted
    f1 = np.divide(2 * tp, denom, out=np.zeros_like(tp), where=denom > 0)
    return float(f1[support > 0].mean())


def per_label_f1(y_true, y_pred) -> dict:
    scores = f1_score(y_true, y_pred, labels=list(range(NUM_LABELS)), average=None, zero_division=0)
    return {LABELS[i]: float(s) for i, s in enumerate(scores)}


def per_group_f1(y_true, y_pred, groups) -> dict:
    y_true, y_pred, groups = map(np.asarray, (y_true, y_pred, groups))
    return {g: f1_scores(y_true[groups == g], y_pred[groups == g]) for g in sorted(set(groups))}


def confusion(y_true, y_pred) -> list:
    return confusion_matrix(y_true, y_pred, labels=list(range(NUM_LABELS))).tolist()


def bootstrap_ci(y_true, y_pred, metric: str = "macro", n_boot: int = 1000, seed: int = 0) -> tuple:
    """95% percentile bootstrap CI of macro-F1 ("macro") or accuracy ("accuracy")."""
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    values = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        if metric == "macro":
            values.append(fast_macro_f1(y_true[idx], y_pred[idx]))
        else:
            values.append(float((y_true[idx] == y_pred[idx]).mean()))
    lo, hi = np.percentile(values, [2.5, 97.5])
    return float(lo), float(hi)


def paired_bootstrap(y_true, pred_a, pred_b, n_boot: int = 2000, seed: int = 0) -> dict:
    """Paired bootstrap test of the macro-F1 difference A - B on the same documents."""
    rng = np.random.default_rng(seed)
    y_true, pred_a, pred_b = map(np.asarray, (y_true, pred_a, pred_b))
    observed = fast_macro_f1(y_true, pred_a) - fast_macro_f1(y_true, pred_b)
    deltas = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        deltas[i] = fast_macro_f1(y_true[idx], pred_a[idx]) - fast_macro_f1(y_true[idx], pred_b[idx])
    p = 2 * min((deltas <= 0).mean(), (deltas >= 0).mean())
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    return {"delta_macro_f1": float(observed), "ci95": [float(lo), float(hi)], "p_value": float(min(p, 1.0))}


def mcnemar_exact(y_true, pred_a, pred_b) -> dict:
    """Exact McNemar test on per-document correctness."""
    y_true, pred_a, pred_b = map(np.asarray, (y_true, pred_a, pred_b))
    a_ok, b_ok = pred_a == y_true, pred_b == y_true
    b, c = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    p = stats.binomtest(b, b + c, 0.5).pvalue if b + c > 0 else 1.0
    return {"a_only_correct": b, "b_only_correct": c, "p_value": float(p)}


def holm(p_values: list) -> list:
    """Holm-Bonferroni adjusted p-values."""
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adjusted[i] = min(running, 1.0)
    return adjusted.tolist()


def krippendorff_alpha_nominal(a, b) -> float:
    """Krippendorff's alpha (nominal) for two coders and no missing values."""
    a, b = np.asarray(a), np.asarray(b)
    pooled = np.concatenate([a, b])
    n = len(pooled)
    counts = np.bincount(pooled, minlength=NUM_LABELS)
    expected = 1 - (counts * (counts - 1)).sum() / (n * (n - 1))
    observed = (a != b).mean()
    return float(1 - observed / expected) if expected > 0 else 1.0


def agreement(a, b) -> dict:
    return {"percent_agreement": float((np.asarray(a) == np.asarray(b)).mean()),
            "cohen_kappa": float(cohen_kappa_score(a, b)),
            "krippendorff_alpha": krippendorff_alpha_nominal(a, b)}


def expected_calibration_error(confidence, correct, n_bins: int = 15) -> tuple:
    """ECE plus the per-bin (confidence, accuracy, count) table for a reliability diagram."""
    confidence, correct = np.asarray(confidence), np.asarray(correct, dtype=float)
    edges = np.linspace(0, 1, n_bins + 1)
    rows, ece = [], 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence > lo) & (confidence <= hi)
        if mask.any():
            conf, acc = confidence[mask].mean(), correct[mask].mean()
            ece += mask.mean() * abs(acc - conf)
            rows.append({"bin_low": lo, "bin_high": hi, "confidence": conf, "accuracy": acc,
                         "count": int(mask.sum())})
    return float(ece), rows

### 1.3 Gaya visualisasi

Semua gambar memakai satu gaya yang konsisten dan ramah cetak:
- Warna per model **tetap** (tidak berubah antar gambar): teacher = biru, **StaticKD (usulan) = oranye**, fastText = aqua, MiniLM = ungu, varian lain = abu-abu. Palet kategorikal ini sudah divalidasi aman untuk buta warna pada pasangan yang bersebelahan. Pada scatter dengan >3 seri, setiap titik juga diberi label teks langsung, sehingga identitas tidak bergantung pada warna saja.
- Skala sekuensial satu hue (biru muda → biru tua) untuk heatmap.
- Garis tipis, grid samar, tanpa sumbu-y ganda.
- Setiap gambar disimpan sebagai **PNG 300 dpi dan PDF (vektor)** di `results/figures/`, siap dimasukkan ke paper.

In [ ]:
%%writefile iptc/viz.py
"""Shared, print-friendly matplotlib style and colour assignments."""
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from iptc.config import FIGURES_DIR

INK, INK_2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
GRAY = "#898781"

# Colour follows the model family everywhere in the notebook.
MODEL_COLORS = {"teacher": BLUE, "statickd": ORANGE, "fasttext": AQUA, "minilm": VIOLET, "other": GRAY}
SEQ_BLUE = LinearSegmentedColormap.from_list(
    "seq_blue", ["#ffffff", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])


def model_family(name: str) -> str:
    if name.startswith("teacher"):
        return "teacher"
    if name.startswith("minilm"):
        return "minilm"
    if name.startswith("ft_"):
        return "fasttext"
    if name.startswith("static_final"):
        return "statickd"
    return "other"


def model_color(name: str) -> str:
    return MODEL_COLORS[model_family(name)]


def apply_style() -> None:
    mpl.rcParams.update({
        "figure.dpi": 110, "savefig.dpi": 300, "figure.facecolor": "white",
        "axes.facecolor": "white", "axes.edgecolor": AXIS, "axes.labelcolor": INK_2,
        "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlecolor": INK,
        "axes.labelsize": 9.5, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
        "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
        "xtick.color": INK_2, "ytick.color": INK_2, "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
        "legend.fontsize": 8.5, "legend.frameon": False, "lines.linewidth": 2.0,
        "font.family": "sans-serif", "pdf.fonttype": 42, "ps.fonttype": 42,
    })


def savefig(fig, name: str) -> None:
    """Save a figure as 300-dpi PNG and vector PDF for the paper."""
    fig.savefig(FIGURES_DIR / f"{name}.png", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / f"{name}.pdf", bbox_inches="tight")
    print(f"[figure] {FIGURES_DIR / name}.png / .pdf")

In [ ]:
from iptc import config, metrics, viz
from iptc.config import LABELS, SHORT_LABELS, LABEL2ID, NUM_LABELS, RESULTS_DIR, ACCURACY_DIR, TABLES_DIR
viz.apply_style()
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40, "display.width", 200, "display.float_format", "{:.3f}".format)


def save_table(df: pd.DataFrame, name: str, index: bool = False, float_format: str = "%.3f") -> None:
    '''Save a table as CSV and LaTeX (booktabs style) for the paper.'''
    df.to_csv(TABLES_DIR / f"{name}.csv", index=index)
    try:
        latex = df.to_latex(index=index, float_format=float_format.__mod__, escape=True)
        (TABLES_DIR / f"{name}.tex").write_text(latex, encoding="utf8")
    except Exception as exc:  # LaTeX export needs jinja2; CSV is always written
        print("LaTeX export skipped:", exc)
    print(f"[table] {TABLES_DIR / name}.csv / .tex")

print("Label IPTC:", len(LABELS))

---
## 2. Data

Riset ini memakai tiga sumber data dengan peran berbeda:

| Data | Peran | Label |
|---|---|---|
| **EMMediaTopic 1.0** (Kuzman & Ljubešić, CLARIN.SI) — 21.000 artikel (hr, sl, ca, el) | Data latih teacher (split `train`) dan **set evaluasi utama** (split `dev`, 1.000 dokumen, tidak pernah dipakai melatih teacher) | GPT-4o |
| **CC-News multibahasa** (arsip 2019–2021, 70 bahasa) | Korpus tak berlabel untuk distilasi + held-out 70 bahasa (situs berita yang tidak ada di data latih) | Logit teacher |
| **Test set LLM 16 bahasa** — 320 artikel dari 16 bahasa di luar data latih teacher | Menguji generalisasi lintas bahasa secara independen dari teacher | Dianotasi LLM dengan prompt Kuzman |

> **Mengapa bukan test set manusia Kuzman?** Test set manusia (IPTC-test, 1.129 dokumen) tidak dipublikasikan untuk menghindari kontaminasi LLM, dan hanya tersedia atas permintaan ke penulis (taja.kuzman@ijs.si). Evaluasi pada set itu adalah langkah validasi berikutnya yang disarankan (lihat Bagian 10).

### 2.1 Modul data
Modul berikut berisi seluruh logika data: unduhan EMMediaTopic, *sampling* CC-News per bahasa, pemisahan held-out **berdasarkan domain**, penambahan data untuk eksperimen skala data, *sampling* dan penggabungan anotasi test set LLM, serta perhitungan statistik deskriptif.

Tiga keputusan desain penting pada *sampling* CC-News:
1. **Konsistensi label bahasa** — hanya dokumen yang bahasanya sama menurut metadata dan fastText-LID.
2. **Batas dokumen per domain** — file CC-News dikelompokkan per situs, sehingga tanpa batas satu situs besar mendominasi. Dokumen diambil dari 3 tahun crawl (2019–2021) agar sumbernya lebih beragam.
3. **Held-out per domain** — seluruh dokumen dari sebagian situs dipisahkan untuk evaluasi, sehingga held-out mengukur generalisasi ke **situs berita yang belum pernah dilihat**, bukan hanya ke dokumen baru dari situs yang sama.

In [ ]:
%%writefile iptc/data.py
"""Data loading, multilingual CC-News sampling, the LLM test set and descriptive statistics."""
import gzip
import json
import random
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import requests

from iptc.config import (EMM_DEV_PATH, EMMEDIATOPIC_PATH, EMMEDIATOPIC_URL, LABEL2ID, LABELS,
                         LLM_TEST_PATH, MAX_WORDS, MIN_WORDS, PROCESSED_DIR, RAW_DIR, SMOKE_TEST,
                         TEACHER_DIR, truncate_words)

# ------------------------------------------------------------------------------ EMMediaTopic
def download_emmediatopic() -> Path:
    """Download EMMediaTopic 1.0 from CLARIN.SI (75 MB; the server can be slow)."""
    if EMMEDIATOPIC_PATH.exists():
        return EMMEDIATOPIC_PATH
    tmp = EMMEDIATOPIC_PATH.with_suffix(".part")
    with requests.get(EMMEDIATOPIC_URL, stream=True, timeout=120) as resp:
        resp.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in resp.iter_content(1 << 20):
                f.write(chunk)
    tmp.rename(EMMEDIATOPIC_PATH)
    return EMMEDIATOPIC_PATH


def load_emmediatopic(split: str | None = None) -> pd.DataFrame:
    """Columns: document_id, lang, text, label (GPT-4o), label_id, split.

    The 1,000-document dev split was never used to train the teacher; it is the clean held-out
    set. Without the raw file (REPORT mode), the dev split is read from emm_dev.jsonl.
    """
    if EMMEDIATOPIC_PATH.exists():
        df = pd.read_json(EMMEDIATOPIC_PATH, lines=True).rename(columns={"GPT-IPTC-label": "label"})
        df["text"] = df["text"].map(truncate_words)
        df["label_id"] = df["label"].map(LABEL2ID)
    elif split == "dev" and EMM_DEV_PATH.exists():
        return pd.read_json(EMM_DEV_PATH, lines=True)
    else:
        raise FileNotFoundError("EMMediaTopic not found: run download_emmediatopic() first")
    if split is not None:
        df = df[df["split"] == split].reset_index(drop=True)
    return df


# ------------------------------------------------------------------------------ CC-News
CCNEWS_URL = "https://huggingface.co/datasets/CloverSearch/cc-news-mutlilingual/resolve/main/{}/{}.jsonl.gz"
# XLM-R languages with a reasonable amount of news in CC-News.
CCNEWS_LANGS = [
    "af", "am", "ar", "az", "be", "bg", "bn", "bs", "ca", "cs", "cy", "da", "de", "el", "en",
    "es", "et", "eu", "fa", "fi", "fr", "gl", "gu", "he", "hi", "hr", "hu", "hy", "id", "is",
    "it", "ja", "ka", "kk", "kn", "ko", "ky", "lt", "lv", "mk", "ml", "mn", "mr", "ms", "my",
    "ne", "nl", "no", "or", "pa", "pl", "ps", "pt", "ro", "ru", "sk", "sl", "so", "sq", "sr",
    "sv", "sw", "ta", "te", "th", "tl", "tr", "uk", "ur", "vi", "zh",
]
SHARDED = {"en": "en00", "es": "es00"}          # very large languages: the first shard suffices
NO_SPACE_LANGS = {"ja", "zh", "th", "my", "km", "lo"}   # length measured in characters
MIN_CHARS_NO_SPACE, MAX_CHARS_NO_SPACE = 200, 2000

if SMOKE_TEST:
    CCNEWS_LANGS = ["sl", "id", "ja"]
    YEARS, MAX_COMPRESSED_BYTES, MAX_PARSED_DOCS = ["2021"], 4_000_000, 3_000
else:
    YEARS, MAX_COMPRESSED_BYTES, MAX_PARSED_DOCS = ["2019", "2020", "2021"], 50_000_000, 25_000
MAX_DOCS_PER_DOMAIN = 150
CANDIDATE_DIR = RAW_DIR / "ccnews_candidates"
CCNEWS_DIR = PROCESSED_DIR / "ccnews"


class _LimitReached(Exception):
    pass


class _ByteCounter:
    """File-like wrapper that stops reading after `limit` compressed bytes."""

    def __init__(self, raw, limit: int):
        self.raw, self.limit, self.read_bytes = raw, limit, 0

    def read(self, size=-1):
        if self.read_bytes >= self.limit:
            raise _LimitReached
        chunk = self.raw.read(size)
        self.read_bytes += len(chunk)
        return chunk


def _build_text(record: dict) -> str:
    """Title + main text, mirroring the MaCoCu texts of EMMediaTopic (title on the first line)."""
    title = (record.get("title") or "").strip()
    body = (record.get("maintext") or "").strip()
    if title and title != "None" and not body.startswith(title):
        return f"{title}\n\n{body}"
    return body


def _long_enough(text: str, lang: str) -> bool:
    return len(text) >= MIN_CHARS_NO_SPACE if lang in NO_SPACE_LANGS else len(text.split()) >= MIN_WORDS


def _truncate(text: str, lang: str) -> str:
    return text[:MAX_CHARS_NO_SPACE] if lang in NO_SPACE_LANGS else truncate_words(text, MAX_WORDS)


def _stream_file(url: str, lang: str) -> list:
    docs, parsed = [], 0
    with requests.get(url, stream=True, timeout=60) as resp:
        resp.raise_for_status()
        try:
            with gzip.GzipFile(fileobj=_ByteCounter(resp.raw, MAX_COMPRESSED_BYTES)) as gz:
                for line in gz:
                    parsed += 1
                    if parsed > MAX_PARSED_DOCS:
                        break
                    try:
                        record = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    # declared language and fastText-LID must agree
                    if record.get("language") != lang or record.get("fasttext_language") != lang:
                        continue
                    text = _build_text(record)
                    if _long_enough(text, lang):
                        docs.append({"text": _truncate(text, lang), "lang": lang,
                                     "domain": record.get("source_domain") or "unknown",
                                     "url": record.get("url")})
        except (EOFError, OSError, _LimitReached):
            pass  # a partial stream is fine: only a sample is needed
    return docs


def stream_language(lang: str) -> list:
    docs = []
    for year in YEARS:
        try:
            docs.extend(_stream_file(CCNEWS_URL.format(year, SHARDED.get(lang, lang)), lang))
        except requests.HTTPError:
            continue  # language missing for that year
    return docs


def load_or_stream(lang: str) -> list:
    """Raw candidates are cached (gzip) so that re-splitting never needs a new download."""
    cache = CANDIDATE_DIR / f"{lang}.candidates.jsonl.gz"
    if cache.exists():
        with gzip.open(cache, "rt", encoding="utf8") as f:
            return [json.loads(line) for line in f]
    docs = stream_language(lang)
    CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)
    with gzip.open(cache, "wt", encoding="utf8") as f:
        for d in docs:
            f.write(json.dumps(d, ensure_ascii=False) + "\n")
    return docs


def domain_split(docs: list, per_lang: int, heldout: int, seed: int) -> tuple:
    """Deduplicate, cap docs per domain, assign whole domains to train or held-out.

    Held-out never exceeds 20% of a language's documents; with < 3 domains a document-level
    split is used instead.
    """
    rng = random.Random(seed)
    by_domain, seen = defaultdict(list), set()
    for d in docs:
        key = d["text"][:300]
        if key not in seen:
            seen.add(key)
            by_domain[d["domain"]].append(d)
    domains = list(by_domain)
    rng.shuffle(domains)
    for dom in domains:
        rng.shuffle(by_domain[dom])
        by_domain[dom] = by_domain[dom][:MAX_DOCS_PER_DOMAIN]
    n_total = sum(len(v) for v in by_domain.values())
    target = min(heldout, int(0.2 * n_total))
    if len(domains) < 3:
        pool = [d for dom in domains for d in by_domain[dom]]
        rng.shuffle(pool)
        return pool[target:target + per_lang], pool[:target]
    held, train = [], []
    for dom in domains:
        (held if len(held) < target else train).extend(by_domain[dom])
    rng.shuffle(train)
    return train[:per_lang], held[:target]


def _write_jsonl(path: Path, rows: list) -> None:
    with open(path, "w", encoding="utf8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def build_ccnews(per_lang: int = 3000, heldout: int = 300, seed: int = 13, workers: int = 8) -> dict:
    """Download/sample CC-News for every language; returns {lang: counts}."""
    CCNEWS_DIR.mkdir(parents=True, exist_ok=True)
    stats = {}
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(load_or_stream, lang): lang for lang in CCNEWS_LANGS}
        for fut in as_completed(futures):
            lang = futures[fut]
            try:
                docs = fut.result()
            except Exception as exc:  # a network error should not stop the other languages
                print(f"[{lang}] failed: {exc}")
                continue
            train, held = domain_split(docs, per_lang, heldout, seed)
            _write_jsonl(CCNEWS_DIR / f"{lang}.train.jsonl", train)
            _write_jsonl(CCNEWS_DIR / f"{lang}.heldout.jsonl", held)
            stats[lang] = {"candidates": len(docs), "train": len(train), "heldout": len(held)}
            print(f"[{lang}] candidates={len(docs)} train={len(train)} heldout={len(held)}", flush=True)
    (CCNEWS_DIR / "stats.json").write_text(json.dumps(dict(sorted(stats.items())), indent=1))
    return stats


def expand_ccnews(extra: int = 5000, seed: int = 21, max_per_domain: int = 400) -> int:
    """Extra training docs from the cached candidates (never from held-out domains)."""
    rng, total = random.Random(seed), 0
    for cache in sorted(CANDIDATE_DIR.glob("*.candidates.jsonl.gz")):
        lang = cache.name.split(".")[0]
        read = lambda p: [json.loads(x) for x in open(p, encoding="utf8")] if p.exists() else []
        train, held = read(CCNEWS_DIR / f"{lang}.train.jsonl"), read(CCNEWS_DIR / f"{lang}.heldout.jsonl")
        held_domains = {d["domain"] for d in held}
        seen = {d["text"][:300] for d in train + held}
        per_domain = Counter(d["domain"] for d in train)
        with gzip.open(cache, "rt", encoding="utf8") as f:
            candidates = [json.loads(line) for line in f]
        rng.shuffle(candidates)
        out = []
        for d in candidates:
            key = d["text"][:300]
            if d["domain"] in held_domains or key in seen or per_domain[d["domain"]] >= max_per_domain:
                continue
            seen.add(key)
            per_domain[d["domain"]] += 1
            out.append(d)
            if len(out) >= extra:
                break
        _write_jsonl(CCNEWS_DIR / f"{lang}.train_extra.jsonl", out)
        total += len(out)
    return total


def load_ccnews(split: str) -> pd.DataFrame:
    files = sorted(CCNEWS_DIR.glob(f"*.{split}.jsonl"))
    frames = [pd.read_json(f, lines=True) for f in files if f.stat().st_size > 0]
    return pd.concat(frames, ignore_index=True)


# ------------------------------------------------------------------------------ LLM test set
LLM_TEST_LANGS = ["en", "de", "fr", "es", "pt", "it", "ru", "pl", "tr", "ar", "zh", "ja", "id", "vi", "hi", "sw"]
LLM_WORK_DIR = PROCESSED_DIR / "llm_test_work"


def sample_llm_test(per_lang: int = 20, seed: int = 7) -> pd.DataFrame:
    """Held-out CC-News docs of 16 languages outside the teacher's training data, spread
    round-robin over news domains. No model prediction is attached (independent annotation)."""
    rows = []
    for lang in LLM_TEST_LANGS:
        path = CCNEWS_DIR / f"{lang}.heldout.jsonl"
        if not path.exists():
            continue
        df = pd.read_json(path, lines=True).sample(frac=1.0, random_state=seed)
        df = df.assign(rank=df.groupby("domain").cumcount()).sort_values("rank", kind="stable")
        rows.append(df.head(per_lang))
    df = pd.concat(rows, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    df["doc_id"] = [f"d{i:03d}" for i in range(len(df))]
    LLM_WORK_DIR.mkdir(parents=True, exist_ok=True)
    df.drop(columns=["rank"]).to_json(LLM_WORK_DIR / "sample.jsonl", orient="records", lines=True,
                                      force_ascii=False)
    return df


def merge_llm_labels(labels: dict) -> pd.DataFrame:
    """labels: {doc_id: IPTC label name}. Writes data/processed/llm_test.jsonl."""
    df = pd.read_json(LLM_WORK_DIR / "sample.jsonl", lines=True)
    df["label"] = df["doc_id"].map(labels)
    df = df[df["label"].isin(LABEL2ID)].copy()
    df["label_id"] = df["label"].map(LABEL2ID)
    df[["doc_id", "lang", "domain", "text", "label", "label_id"]].to_json(
        LLM_TEST_PATH, orient="records", lines=True, force_ascii=False)
    return df


def load_llm_test() -> pd.DataFrame:
    return pd.read_json(LLM_TEST_PATH, lines=True)


# ------------------------------------------------------------------------------ statistics
def _n_words(texts: pd.Series) -> list:
    return texts.str.split().str.len().astype(int).tolist()


def compute_data_stats(sample_per_lang: int = 300, seed: int = 0) -> dict:
    """Descriptive statistics of every corpus (feeds the figures/tables of Section 2 and 4)."""
    rng = np.random.default_rng(seed)
    stats = {}
    emm = load_emmediatopic()
    stats["emm_counts"] = emm.groupby(["split", "lang", "label"]).size().rename("n").reset_index().to_dict("records")
    stats["emm_words"] = {s: _n_words(g["text"]) for s, g in emm.groupby("split")}

    per_lang = {}
    for split in ("train", "train_extra", "heldout"):
        for f in sorted(CCNEWS_DIR.glob(f"*.{split}.jsonl")):
            lang = f.name.split(".")[0]
            df = pd.read_json(f, lines=True) if f.stat().st_size > 0 else pd.DataFrame(columns=["text", "domain"])
            entry = per_lang.setdefault(lang, {})
            entry[split] = int(len(df))
            entry[f"{split}_domains"] = int(df["domain"].nunique()) if len(df) else 0
            if split == "train" and len(df):
                sample = df["text"].sample(min(len(df), 200), random_state=seed)
                entry["words_sample"] = _n_words(sample)
    stats["ccnews_per_lang"] = per_lang

    teacher = {}
    for name in ("emmediatopic", "ccnews_train", "ccnews_train_extra", "ccnews_heldout"):
        path = TEACHER_DIR / f"{name}.parquet"
        if not path.exists():
            continue
        df = pd.read_parquet(path, columns=["lang", "teacher_logits", "teacher_label_id"])
        logits = np.stack(df["teacher_logits"].to_numpy())
        probs = np.exp(logits - logits.max(1, keepdims=True))
        probs /= probs.sum(1, keepdims=True)
        df["confidence"] = probs.max(1)
        entry = {"label_counts": {LABELS[i]: int(c) for i, c in df["teacher_label_id"].value_counts().items()},
                 "confidence_mean_per_lang": df.groupby("lang")["confidence"].mean().round(4).to_dict()}
        if name == "ccnews_heldout":
            entry["confidence_sample_per_lang"] = {
                lang: g["confidence"].sample(min(len(g), sample_per_lang), random_state=seed).round(4).tolist()
                for lang, g in df.groupby("lang")}
        teacher[name] = entry
    stats["teacher"] = teacher

    if LLM_TEST_PATH.exists():
        llm = load_llm_test()
        stats["llm_test_counts"] = llm.groupby(["lang", "label"]).size().rename("n").reset_index().to_dict("records")
    return stats

In [ ]:
from iptc import data
from iptc.config import DATA_STATS_PATH, EMMEDIATOPIC_PATH, LLM_TEST_PATH

if MODE == "FULL":
    t0 = time.time()
    data.download_emmediatopic()
    print(f"EMMediaTopic: {EMMEDIATOPIC_PATH} ({EMMEDIATOPIC_PATH.stat().st_size / 2**20:.1f} MB, "
          f"{time.time() - t0:.0f} s)")
    emm = data.load_emmediatopic()
    if SMOKE_TEST:  # keep the dev split intact-ish but small, and a small train split
        emm = pd.concat([emm[emm.split == "train"].sample(1000, random_state=0),
                         emm[emm.split == "dev"].sample(200, random_state=0)])
        emm.rename(columns={"label": "GPT-IPTC-label"}).drop(columns=["label_id"]).to_json(
            EMMEDIATOPIC_PATH, orient="records", lines=True, force_ascii=False)
        emm = data.load_emmediatopic()
    data.load_emmediatopic("dev").to_json(config.EMM_DEV_PATH, orient="records", lines=True, force_ascii=False)
    print(emm.groupby(["split", "lang"]).size().unstack())
else:
    print("REPORT: statistik EMMediaTopic dimuat dari", DATA_STATS_PATH)

### 2.2 Korpus CC-News multibahasa (data distilasi)

Teacher dapat melabeli data sebanyak apa pun secara gratis (hanya butuh GPU). Karena itu kita mengumpulkan korpus berita **tak berlabel** yang besar dan beragam bahasa, bukan hanya 15 ribu artikel 4 bahasa seperti Kuzman.

- Sumber: `CloverSearch/cc-news-mutlilingual` (Common Crawl News, dipisah per bahasa dan tahun).
- Per bahasa: maks. 3.000 dokumen latih + maks. 300 dokumen held-out (≤ 20% dokumen bahasa itu), maks. 150 dokumen per situs.
- Praproses sama dengan EMMediaTopic: judul + isi, ≥ 75 kata (≥ 200 karakter untuk aksara tanpa spasi: ja, zh, th, my), dipotong ke 512 kata (2.000 karakter).
- **Eksperimen skala data**: 5.000 dokumen tambahan per bahasa dari kandidat yang sama, tidak pernah dari domain held-out.

Estimasi di Colab: ± 20–40 menit unduhan (paralel, ± 5 GB dibaca).

In [ ]:
if MODE == "FULL":
    t0 = time.time()
    per_lang, heldout = (300, 60) if SMOKE_TEST else (3000, 300)
    cc_stats = data.build_ccnews(per_lang=per_lang, heldout=heldout)
    print(f"\n{len(cc_stats)} bahasa, train={sum(v['train'] for v in cc_stats.values())}, "
          f"heldout={sum(v['heldout'] for v in cc_stats.values())}  ({(time.time() - t0) / 60:.1f} menit)")
    if RUN_DATA_SCALING:
        n_extra = data.expand_ccnews(extra=500 if SMOKE_TEST else 5000)
        print("Dokumen tambahan (eksperimen skala data):", n_extra)
else:
    print("REPORT: statistik CC-News dimuat dari", DATA_STATS_PATH)

### 2.3 Test set LLM 16 bahasa

Test set ini menjawab pertanyaan yang tidak bisa dijawab data dev: **apakah model tetap benar pada bahasa yang tidak ada di data latih teacher?**

- 20 dokumen × 16 bahasa (en, de, fr, es, pt, it, ru, pl, tr, ar, zh, ja, id, vi, hi, sw) dari split **held-out** CC-News (situs tidak terlihat), disebar merata antar situs.
- **Anotasi**: LLM (Claude) dengan **prompt dan deskripsi label yang persis sama** dengan yang dipakai Kuzman & Ljubešić untuk GPT-4o (Lampiran B paper mereka). Anotator tidak melihat prediksi model mana pun.
- Pada riset asli, anotasi dilakukan oleh Claude (dijalankan sebagai agen yang membaca setiap artikel). Sel opsional di bawah menyediakan reproduksi via Claude API (`claude-opus-5`, *structured output* JSON sehingga jawaban selalu berupa salah satu label valid).
- Validitas anotasi diuji di Bagian 3.4 dengan mengukur kesepakatannya dengan teacher, dibandingkan dengan kesepakatan manusia–GPT-4o yang dilaporkan Kuzman.

> Biaya API: ± 320 permintaan × ~1.500 token input → di bawah US$5 pada harga Opus. Aktifkan hanya bila `ANNOTATE_WITH_CLAUDE = True` dan API key (`ANTHROPIC_API_KEY`) tersedia di Colab Secrets. Sel ini memakai *server-side fallback* (`fallbacks="default"`): bila permintaan ditolak oleh filter keamanan, API otomatis mengulang di model cadangan.

In [ ]:
%%writefile iptc/llm_annotation.py
"""Zero-shot IPTC annotation with Claude, using Kuzman & Ljubesic's GPT-4o prompt (Appendix B)."""
import json

LABEL_DESCRIPTIONS = {
    "disaster, accident and emergency incident": "man-made or natural events resulting in injuries, death or damage, e.g., explosions, transport accidents, famine, drowning, natural disasters, emergency planning and response.",
    "human interest": "news about life and behavior of royalty and celebrities, news about obtaining awards, ceremonies (graduation, wedding, funeral, celebration of launching something), birthdays and anniversaries, and news about silly or stupid human errors.",
    "politics": "news about local, regional, national and international exercise of power, including news about election, fundamental rights, government, non-governmental organisations, political crises, non-violent international relations, public employees, government policies.",
    "education": "all aspects of furthering knowledge, formally or informally, including news about schools, curricula, grading, remote learning, teachers and students.",
    "crime, law and justice": "news about committed crime and illegal activities, the system of courts, law and law enforcement (e.g., judges, lawyers, trials, punishments of offenders).",
    "economy, business and finance": "news about companies, products and services, any kind of industries, national economy, international trading, banks, (crypto)currency, business and trade societies, economic trends and indicators (inflation, employment statistics, GDP, mortgages, ...), international economic institutions, utilities (electricity, heating, waste management, water supply).",
    "conflict, war and peace": "news about terrorism, wars, wars victims, cyber warfare, civil unrest (demonstrations, riots, rebellions), peace talks and other peace activities.",
    "arts, culture, entertainment and media": "news about cinema, dance, fashion, hairstyle, jewellery, festivals, literature, music, theatre, TV shows, painting, photography, woodworking, art exhibitions, libraries and museums, language, cultural heritage, news media, radio and television, social media, influencers, and disinformation.",
    "labour": "news about employment, employment legislation, employees and employers, commuting, parental leave, volunteering, wages, social security, labour market, retirement, unemployment, unions.",
    "weather": "news about weather forecasts, weather phenomena and weather warning.",
    "religion": "news about religions, cults, religious conflicts, relations between religion and government, churches, religious holidays and festivals, religious leaders and rituals, and religious texts.",
    "society": "news about social interactions (e.g., networking), demographic analyses, population census, discrimination, efforts for inclusion and equity, emigration and immigration, communities of people and minorities (LGBTQ, older people, children, indigenous people, etc.), homelessness, poverty, societal problems (addictions, bullying), ethical issues (suicide, euthanasia, sexual behavior) and social services and charity, relationships (dating, divorce, marriage), family (family planning, adoption, abortion, contraception, pregnancy, parenting).",
    "health": "news about diseases, injuries, mental health problems, health treatments, diets, vaccines, drugs, government health care, hospitals, medical staff, health insurance.",
    "environment": "news about climate change, energy saving, sustainability, pollution, population growth, natural resources, forests, mountains, bodies of water, ecosystem, animals, flowers and plants.",
    "lifestyle and leisure": "news about hobbies, clubs and societies, games, lottery, enthusiasm about food or drinks, car/motorcycle lovers, public holidays, leisure venues (amusement parks, cafes, bars, restaurants, etc.), exercise and fitness, outdoor recreational activities (e.g., fishing, hunting), travel and tourism, mental well-being, parties, maintaining and decorating house and garden.",
    "science and technology": "news about natural sciences and social sciences, mathematics, technology and engineering, scientific institutions, scientific research, scientific publications and innovation.",
    "sport": "news about sports that can be executed in competitions - basketball, football, swimming, athletics, chess, dog racing, diving, golf, gymnastics, martial arts, climbing, etc.; sport achievements, sport events, sport organisation, sport venues (stadiums, gymnasiums, ...), referees, coaches, sport clubs, drug use in sport.",
}
# Integer ids in the order of Kuzman's prompt dictionary.
PROMPT_LABELS = list(LABEL_DESCRIPTIONS)
LABEL_DICT = {f"{name} - {desc}": i for i, (name, desc) in enumerate(LABEL_DESCRIPTIONS.items())}

SYSTEM_PROMPT = f"""### Task
Your task is to classify the provided text into a topic label, meaning that you need to recognize what is the topic of the text. You will be provided with a news text, delimited by single quotation marks. Always provide a label, even if you are not sure.

### Output format
Return a valid JSON dictionary with the following key: 'topic' and a value should be an integer which represents one of the labels according to the following dictionary: {LABEL_DICT}."""

OUTPUT_SCHEMA = {
    "type": "object",
    "properties": {"topic": {"type": "integer", "enum": list(range(len(PROMPT_LABELS)))}},
    "required": ["topic"],
    "additionalProperties": False,
}


def annotate(texts: list, model: str = "claude-opus-5") -> list:
    """Return one IPTC label name (or None on refusal) per text."""
    import anthropic

    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY
    labels = []
    for text in texts:
        response = client.beta.messages.create(
            model=model,
            max_tokens=4000,
            betas=["server-side-fallback-2026-07-01"],
            fallbacks="default",  # re-run on a fallback model if the request is declined
            system=SYSTEM_PROMPT,
            output_config={"format": {"type": "json_schema", "schema": OUTPUT_SCHEMA}},
            messages=[{"role": "user", "content": f"Text: '{text}'"}],
        )
        if response.stop_reason == "refusal":
            labels.append(None)
            continue
        answer = next(b.text for b in response.content if b.type == "text")
        labels.append(PROMPT_LABELS[json.loads(answer)["topic"]])
    return labels

In [ ]:
if MODE == "FULL":
    if LLM_TEST_PATH.exists():
        print("Test set LLM sudah ada:", LLM_TEST_PATH)
    elif ANNOTATE_WITH_CLAUDE:
        if IN_COLAB:
            from google.colab import userdata
            os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        from iptc import llm_annotation
        sample = data.sample_llm_test(per_lang=20)
        t0 = time.time()
        labels = llm_annotation.annotate(sample["text"].str[:3000].tolist())
        data.merge_llm_labels(dict(zip(sample["doc_id"], labels)))
        print(f"Dianotasi {sum(l is not None for l in labels)}/{len(labels)} dok ({time.time() - t0:.0f} s)")
    else:
        raise FileNotFoundError(
            f"Unggah llm_test.jsonl (ada di bundle) ke {LLM_TEST_PATH} atau set ANNOTATE_WITH_CLAUDE = True")
llm_test = data.load_llm_test()
print(f"Test set LLM: {len(llm_test)} dokumen, {llm_test['lang'].nunique()} bahasa")
llm_test.head(3)[["doc_id", "lang", "domain", "label"]]

### 2.4 Statistik deskriptif data

Statistik berikut dihitung satu kali dari seluruh korpus (pada mode FULL setelah pelabelan teacher di Bagian 4 selesai, statistik dihitung ulang agar mencakup distribusi label teacher) dan disimpan di `results/data_stats.json`.

In [ ]:
if MODE == "FULL" and not DATA_STATS_PATH.exists():
    DATA_STATS_PATH.write_text(json.dumps(data.compute_data_stats()))
DATA_STATS = json.loads(DATA_STATS_PATH.read_text())

emm_counts = pd.DataFrame(DATA_STATS["emm_counts"])
cc = pd.DataFrame(DATA_STATS["ccnews_per_lang"]).T.fillna(0)
for col in ("train", "train_extra", "heldout", "train_domains", "heldout_domains"):
    if col not in cc:
        cc[col] = 0
cc[["train", "train_extra", "heldout", "train_domains", "heldout_domains"]] = \
    cc[["train", "train_extra", "heldout", "train_domains", "heldout_domains"]].astype(int)

summary = pd.DataFrame([
    {"Korpus": "EMMediaTopic train", "Dokumen": int(emm_counts.query("split == 'train'").n.sum()),
     "Bahasa": emm_counts.query("split == 'train'").lang.nunique(), "Label": "GPT-4o + logit teacher",
     "Peran": "distilasi"},
    {"Korpus": "EMMediaTopic dev", "Dokumen": int(emm_counts.query("split == 'dev'").n.sum()),
     "Bahasa": emm_counts.query("split == 'dev'").lang.nunique(), "Label": "GPT-4o", "Peran": "evaluasi utama"},
    {"Korpus": "CC-News train", "Dokumen": int(cc.train.sum()), "Bahasa": int((cc.train > 0).sum()),
     "Label": "logit teacher", "Peran": "distilasi"},
    {"Korpus": "CC-News train tambahan", "Dokumen": int(cc.train_extra.sum()),
     "Bahasa": int((cc.train_extra > 0).sum()), "Label": "logit teacher", "Peran": "skala data"},
    {"Korpus": "CC-News held-out (domain baru)", "Dokumen": int(cc.heldout.sum()),
     "Bahasa": int((cc.heldout > 0).sum()), "Label": "logit teacher", "Peran": "fidelity 70 bahasa"},
    {"Korpus": "Test set LLM", "Dokumen": len(llm_test), "Bahasa": llm_test.lang.nunique(),
     "Label": "LLM (prompt Kuzman)", "Peran": "generalisasi lintas bahasa"},
])
save_table(summary, "table1_datasets")
summary

In [ ]:
# Figure: label distribution of EMMediaTopic (GPT-4o labels) and of the LLM test set
emm_label = emm_counts.groupby("label").n.sum().reindex(LABELS).fillna(0)
llm_label = llm_test.label.value_counts().reindex(LABELS).fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
order = np.argsort(emm_label.values)
for ax, series, title in ((axes[0], emm_label / emm_label.sum(), "EMMediaTopic (21k, 4 bahasa, label GPT-4o)"),
                          (axes[1], llm_label / llm_label.sum(), "Test set LLM (320, 16 bahasa)")):
    ax.barh(np.arange(NUM_LABELS), series.values[order] * 100, color=viz.BLUE, height=0.7)
    ax.set_title(title)
    ax.set_xlabel("% dokumen")
    ax.grid(axis="y", visible=False)
axes[0].set_yticks(np.arange(NUM_LABELS), [SHORT_LABELS[i] for i in order])
fig.suptitle("Distribusi label IPTC", fontweight="bold", x=0.01, ha="left")
fig.tight_layout()
viz.savefig(fig, "fig01_label_distribution")
plt.show()

per_lang_split = emm_counts.pivot_table(index="lang", columns="split", values="n", aggfunc="sum")
print("EMMediaTopic per bahasa:\n", per_lang_split)

In [ ]:
# Figure: CC-News documents per language (train / extra / held-out) and number of source domains
cc_sorted = cc.sort_values("train", ascending=False)
x = np.arange(len(cc_sorted))
fig, axes = plt.subplots(2, 1, figsize=(13, 6.2), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
axes[0].bar(x, cc_sorted.train, color=viz.BLUE, width=0.75, label="train")
axes[0].bar(x, cc_sorted.train_extra, bottom=cc_sorted.train, color=viz.ORANGE, width=0.75,
            label="train tambahan (skala data)", edgecolor="white", linewidth=0.5)
axes[0].set_ylabel("dokumen")
axes[0].legend(loc="upper right")
axes[0].set_title("CC-News multibahasa: jumlah dokumen latih per bahasa")
axes[1].bar(x, cc_sorted.train_domains, color=viz.GRAY, width=0.75)
axes[1].set_ylabel("situs (train)")
axes[1].set_xticks(x, cc_sorted.index, rotation=90, fontsize=7.5)
for ax in axes:
    ax.grid(axis="x", visible=False)
fig.tight_layout()
viz.savefig(fig, "fig02_ccnews_per_language")
plt.show()

print(f"Bahasa dengan >= 1.000 dok latih: {(cc.train >= 1000).sum()} | < 100 dok: {(cc.train < 100).sum()}")
print(f"Median situs per bahasa (train): {cc.train_domains.median():.0f}")

In [ ]:
# Figure: document length (words) — EMMediaTopic vs CC-News
emm_words = np.concatenate([np.asarray(v) for v in DATA_STATS["emm_words"].values()])
cc_words = np.concatenate([np.asarray(v.get("words_sample", [])) for v in DATA_STATS["ccnews_per_lang"].values()])
fig, ax = plt.subplots(figsize=(7, 3.4))
bins = np.linspace(0, 520, 53)
ax.hist(emm_words, bins=bins, histtype="step", color=viz.BLUE, linewidth=2, density=True, label="EMMediaTopic")
ax.hist(cc_words, bins=bins, histtype="step", color=viz.ORANGE, linewidth=2, density=True,
        label="CC-News (sampel, bahasa berspasi)")
ax.set_xlabel("jumlah kata (dipotong di 512)")
ax.set_ylabel("densitas")
ax.set_title("Panjang dokumen")
ax.legend()
viz.savefig(fig, "fig03_document_length")
plt.show()
print(pd.DataFrame({"EMMediaTopic": pd.Series(emm_words).describe(),
                    "CC-News": pd.Series(cc_words).describe()}).T[["count", "mean", "50%", "min", "max"]])

---
## 3. Review model teacher (Kuzman & Ljubešić, 2025)

### 3.1 Profil model
| Aspek | Nilai |
|---|---|
| Model | `classla/multilingual-IPTC-news-topic-classifier` |
| Arsitektur | XLM-RoBERTa-large (24 layer, hidden 1024, 16 head), ±560 juta parameter |
| Data latih | 15.000 artikel EMMediaTopic (hr, sl, ca, el) berlabel GPT-4o |
| Input | teks artikel, dipotong 512 token |
| Skor yang dilaporkan paper | macro-F1 0,746 / micro-F1 0,734 pada test set manusia (1.129 dokumen) |

Riset ini mengevaluasi teacher dalam dua konfigurasi CPU:
1. **PyTorch fp32**, cara pakai yang ditulis di model card (`pipeline(...)`).
2. **ONNX int8** (ekspor `onnx-community`), konfigurasi CPU paling optimal yang tersedia untuk model ini. Tujuannya agar perbandingan efisiensi adil terhadap teacher.

### 3.2 Modul teacher, predictor, dan evaluasi
- `Teacher` mengembalikan **logit 17 kelas** dalam urutan label proyek. Urutan label di `config.id2label` model berbeda, sehingga dipermutasi.
- `load_predictor(spec)` membuat semua model (teacher, fastText, StaticKD, transformer) memiliki antarmuka yang sama `predict(texts) → label_id`, sehingga evaluasi dan benchmark identik untuk semua model.
- `evaluate_model` menghitung metrik pada tiga set evaluasi dan menyimpan **semua prediksi**. Prediksi dibutuhkan untuk uji signifikansi berpasangan di Bagian 6.

In [ ]:
%%writefile iptc/teacher.py
"""The teacher: Kuzman & Ljubesic's XLM-R-large IPTC classifier, plus distillation labelling."""
import time

import numpy as np
import pandas as pd

from iptc.config import LABEL2ID, LABELS, TEACHER_DIR, TEACHER_MODEL_ID


class Teacher:
    """Returns logits in the project-wide label order (config.LABELS)."""

    def __init__(self, device: str | None = None, fp16: bool = True, max_length: int = 512):
        import torch
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID)
        model = AutoModelForSequenceClassification.from_pretrained(TEACHER_MODEL_ID)
        if fp16 and self.device == "cuda":
            model = model.half()
        self.model = model.to(self.device).eval()
        id2label = self.model.config.id2label
        self.order = np.array([LABEL2ID[id2label[i]] for i in range(len(LABELS))])

    def num_parameters(self) -> int:
        return sum(p.numel() for p in self.model.parameters())

    def logits(self, texts: list, batch_size: int = 16) -> np.ndarray:
        """(n_texts, 17) logits; texts are length-sorted to minimise padding."""
        order = np.argsort([len(t) for t in texts])[::-1]
        out = np.zeros((len(texts), len(LABELS)), dtype=np.float32)
        with self.torch.inference_mode():
            for start in range(0, len(texts), batch_size):
                idx = order[start:start + batch_size]
                enc = self.tokenizer([texts[i] for i in idx], truncation=True, max_length=self.max_length,
                                     padding=True, return_tensors="pt").to(self.device)
                model_logits = self.model(**enc).logits.float().cpu().numpy()
                reordered = np.zeros_like(model_logits)
                reordered[:, self.order] = model_logits
                out[idx] = reordered
        return out


def pseudo_label(teacher: Teacher, df: pd.DataFrame, name: str, batch_size: int = 16,
                 shard_size: int = 10_000) -> pd.DataFrame:
    """Annotate `df` with teacher logits; shards are saved so a Colab disconnect can resume."""
    out = TEACHER_DIR / f"{name}.parquet"
    if out.exists():
        return pd.read_parquet(out)
    shard_dir = TEACHER_DIR / f"{name}_shards"
    shard_dir.mkdir(parents=True, exist_ok=True)
    start_time = time.time()
    for i, start in enumerate(range(0, len(df), shard_size)):
        path = shard_dir / f"part{i:04d}.parquet"
        if path.exists():
            continue
        part = df.iloc[start:start + shard_size].copy()
        logits = teacher.logits(part["text"].tolist(), batch_size=batch_size)
        part["teacher_logits"] = list(logits)
        part["teacher_label_id"] = logits.argmax(axis=1)
        part.to_parquet(path, index=False)
        done = min(start + shard_size, len(df))
        rate = done / (time.time() - start_time + 1e-9)
        print(f"[{name}] {done}/{len(df)} docs ({rate:.1f} docs/s)", flush=True)
    full = pd.concat([pd.read_parquet(p) for p in sorted(shard_dir.glob("part*.parquet"))], ignore_index=True)
    full.to_parquet(out, index=False)
    return full

In [ ]:
%%writefile iptc/predictors.py
"""Uniform CPU predictors: every model exposes predict(texts) -> label ids in config.LABELS order.

Specs:  teacher-pytorch | teacher-onnx-int8 | fasttext:<file> | static:<dir> | onnx:<dir> | onnx-int8:<dir>
"""
from pathlib import Path

import numpy as np

from iptc.config import LABEL2ID, LABELS, TEACHER_MODEL_ID, TEACHER_ONNX_ID


def dir_size_mb(path) -> float:
    path = Path(path)
    if path.is_file():
        return path.stat().st_size / 2**20
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 2**20


class TeacherPyTorch:
    def __init__(self, threads: int):
        import torch
        from huggingface_hub import snapshot_download
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        torch.set_num_threads(threads)
        self.torch = torch
        path = snapshot_download(TEACHER_MODEL_ID, allow_patterns=["*.json", "*.safetensors", "*.model"])
        self.tokenizer = AutoTokenizer.from_pretrained(path)
        self.model = AutoModelForSequenceClassification.from_pretrained(path).float().eval()
        self.order = np.array([LABEL2ID[self.model.config.id2label[i]] for i in range(len(LABELS))])
        self.size_mb = dir_size_mb(Path(path) / "model.safetensors")
        self.num_parameters = sum(p.numel() for p in self.model.parameters())

    def predict(self, texts):
        with self.torch.inference_mode():
            enc = self.tokenizer(texts, truncation=True, max_length=512, padding=True, return_tensors="pt")
            return self.order[self.model(**enc).logits.argmax(-1).numpy()]


class OnnxClassifier:
    """ONNX Runtime sequence classifier (int8 teacher and transformer students)."""

    def __init__(self, model_dir: str, onnx_file: str, threads: int, max_length: int = 512):
        import onnxruntime as ort
        from transformers import AutoConfig, AutoTokenizer

        opts = ort.SessionOptions()
        opts.intra_op_num_threads = threads
        opts.inter_op_num_threads = 1
        opts.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        # Busy-waiting worker threads made multi-threaded runs unstable on hybrid CPUs.
        opts.add_session_config_entry("session.intra_op.allow_spinning", "0")
        onnx_path = str(Path(model_dir) / onnx_file)
        self.session = ort.InferenceSession(onnx_path, opts, providers=["CPUExecutionProvider"])
        self.input_names = {i.name for i in self.session.get_inputs()}
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        cfg = AutoConfig.from_pretrained(model_dir)
        self.order = np.array([LABEL2ID[cfg.id2label[i]] for i in range(len(LABELS))])
        self.max_length = max_length
        self.size_mb = dir_size_mb(onnx_path) + (dir_size_mb(onnx_path + "_data")
                                                 if Path(onnx_path + "_data").exists() else 0)
        self.num_parameters = None

    def predict(self, texts):
        enc = self.tokenizer(texts, truncation=True, max_length=self.max_length, padding=True, return_tensors="np")
        feeds = {k: v.astype(np.int64) for k, v in enc.items() if k in self.input_names}
        return self.order[self.session.run(None, feeds)[0].argmax(-1)]


def load_predictor(spec: str, threads: int = 1):
    kind, _, path = spec.partition(":")
    if kind == "teacher-pytorch":
        return TeacherPyTorch(threads)
    if kind == "teacher-onnx-int8":
        from huggingface_hub import snapshot_download
        model_dir = snapshot_download(TEACHER_ONNX_ID, allow_patterns=["*.json", "onnx/model_quantized.onnx"])
        return OnnxClassifier(model_dir, "onnx/model_quantized.onnx", threads)
    if kind == "onnx":
        return OnnxClassifier(path, "model.onnx", threads, max_length=256)
    if kind == "onnx-int8":
        return OnnxClassifier(path, "model_int8.onnx", threads, max_length=256)
    if kind == "fasttext":
        from iptc.fasttext_student import FastTextPredictor
        return FastTextPredictor(path)
    if kind == "static":
        from iptc.static_student import StaticPredictor
        return StaticPredictor(path)
    raise ValueError(f"Unknown model spec: {spec}")

In [ ]:
%%writefile iptc/evaluation.py
"""Accuracy evaluation on the three evaluation sets; every prediction is stored."""
import json

import numpy as np
import pandas as pd

from iptc import metrics
from iptc.config import ACCURACY_DIR, LABELS, LLM_TEST_PATH, TEACHER_DIR
from iptc.data import load_emmediatopic


def evaluation_sets(names=("dev", "ccnews", "llm_test")) -> dict:
    """dev: GPT-4o labels | ccnews: teacher labels on unseen domains | llm_test: LLM labels."""
    sets = {}
    if "dev" in names:
        sets["dev"] = (load_emmediatopic("dev"), "label_id")
    heldout = TEACHER_DIR / "ccnews_heldout.parquet"
    if "ccnews" in names and heldout.exists():
        sets["ccnews"] = (pd.read_parquet(heldout, columns=["text", "lang", "teacher_label_id"]), "teacher_label_id")
    if "llm_test" in names and LLM_TEST_PATH.exists():
        sets["llm_test"] = (pd.read_json(LLM_TEST_PATH, lines=True), "label_id")
    return sets


def predict_all(predictor, texts: list, batch_size: int = 32) -> np.ndarray:
    return np.concatenate([predictor.predict(texts[i:i + batch_size]) for i in range(0, len(texts), batch_size)])


def evaluate_set(predictor, df: pd.DataFrame, label_col: str) -> dict:
    y_pred = predict_all(predictor, df["text"].tolist())
    y_true = df[label_col].to_numpy()
    return {
        **metrics.f1_scores(y_true, y_pred),
        "macro_f1_ci95": list(metrics.bootstrap_ci(y_true, y_pred)),
        "per_language": metrics.per_group_f1(y_true, y_pred, df["lang"].to_numpy()),
        "per_label_f1": metrics.per_label_f1(y_true, y_pred),
        "confusion": metrics.confusion(y_true, y_pred),
        "predictions": [LABELS[i] for i in y_pred],
    }


def evaluate_model(spec: str, name: str, sets=("dev", "ccnews", "llm_test"), threads: int = 4,
                   overwrite: bool = False) -> dict:
    out = ACCURACY_DIR / f"{name}.json"
    if out.exists() and not overwrite:
        return json.loads(out.read_text(encoding="utf8"))
    from iptc.predictors import load_predictor

    predictor = load_predictor(spec, threads)
    results = {"model": name, "spec": spec}
    for set_name, (df, label_col) in evaluation_sets(sets).items():
        results[set_name] = evaluate_set(predictor, df, label_col)
        print(f"  [{name} | {set_name}] macro-F1={results[set_name]['macro_f1']:.3f} "
              f"micro-F1={results[set_name]['micro_f1']:.3f}")
    out.write_text(json.dumps(results, ensure_ascii=False), encoding="utf8")
    return results


def load_results(name: str) -> dict:
    return json.loads((ACCURACY_DIR / f"{name}.json").read_text(encoding="utf8"))


def predictions(name: str, split: str) -> np.ndarray:
    from iptc.config import LABEL2ID
    return np.array([LABEL2ID[p] for p in load_results(name)[split]["predictions"]])

### 3.3 Akurasi teacher

Evaluasi teacher di **CPU** (fp32 PyTorch dan ONNX int8) pada data dev dan test set LLM. Di Colab, gunakan GPU hanya jika ingin mempercepat. Hasil akurasinya identik, hanya kecepatan yang berbeda, dan kecepatan diukur terpisah di Bagian 7.

**Cara membaca hasil:**
- Skor dev lebih tinggi dari 0,746 di paper karena label dev adalah **label GPT-4o**, yaitu label yang memang ditiru teacher, bukan label manusia. Paper Kuzman melaporkan kesepakatan GPT-4o dengan manusia hanya ≈ 0,73 (Krippendorff's α).
- Skor test set LLM menunjukkan kemampuan **lintas bahasa (zero-shot cross-lingual)** teacher pada 16 bahasa yang tidak ada di data fine-tuning-nya.

In [ ]:
from iptc import evaluation

if MODE == "FULL":
    for spec, name in (("teacher-pytorch", "teacher_xlmr_large"), ("teacher-onnx-int8", "teacher_xlmr_large_onnx_int8")):
        t0 = time.time()
        evaluation.evaluate_model(spec, name, sets=("dev", "llm_test"), threads=os.cpu_count())
        print(f"{name}: {time.time() - t0:.0f} s")

teacher = evaluation.load_results("teacher_xlmr_large")
teacher_onnx = evaluation.load_results("teacher_xlmr_large_onnx_int8")
rows = []
for name, res in (("Teacher PyTorch fp32", teacher), ("Teacher ONNX int8", teacher_onnx)):
    for split, split_name in (("dev", "dev (4 bahasa, label GPT-4o)"), ("llm_test", "test LLM (16 bahasa lain)")):
        r = res[split]
        rows.append({"Model": name, "Set": split_name, "Macro-F1": r["macro_f1"],
                     "CI95 bawah": r["macro_f1_ci95"][0], "CI95 atas": r["macro_f1_ci95"][1],
                     "Micro-F1": r["micro_f1"], "n": r["n"]})
teacher_table = pd.DataFrame(rows)
save_table(teacher_table, "table2_teacher_accuracy")
teacher_table

In [ ]:
# Per-language scores of the teacher
per_lang_rows = []
for split in ("dev", "llm_test"):
    for lang, r in teacher[split]["per_language"].items():
        per_lang_rows.append({"set": split, "bahasa": lang, "macro_f1": r["macro_f1"], "akurasi": r["accuracy"], "n": r["n"]})
teacher_lang = pd.DataFrame(per_lang_rows)
display_cols = teacher_lang.pivot_table(index="bahasa", columns="set", values="akurasi")
print("Akurasi teacher per bahasa (dev: 4 bahasa latih; llm_test: 16 bahasa lain, 20 dok/bahasa):")
display_cols.sort_values(["llm_test", "dev"], ascending=False).T

In [ ]:
# Figure: teacher per-label F1 and normalised confusion matrix (dev)
def plot_confusion(cm, title, ax, cmap=viz.SEQ_BLUE):
    cm = np.asarray(cm, dtype=float)
    norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    im = ax.imshow(norm, cmap=cmap, vmin=0, vmax=1)
    ax.set_xticks(range(NUM_LABELS), SHORT_LABELS, rotation=60, ha="right", fontsize=7.5)
    ax.set_yticks(range(NUM_LABELS), SHORT_LABELS, fontsize=7.5)
    ax.set_xlabel("prediksi")
    ax.set_ylabel("label benar")
    ax.set_title(title)
    ax.grid(False)
    for i in range(NUM_LABELS):
        for j in range(NUM_LABELS):
            if norm[i, j] >= 0.1:
                ax.text(j, i, f"{norm[i, j]:.2f}".lstrip("0"), ha="center", va="center", fontsize=6,
                        color="white" if norm[i, j] > 0.55 else viz.INK)
    return im

fig, axes = plt.subplots(1, 2, figsize=(13, 5.4), gridspec_kw={"width_ratios": [1, 1.35]})
f1 = pd.Series(teacher["dev"]["per_label_f1"]).reindex(LABELS)
order = np.argsort(f1.values)
axes[0].barh(np.arange(NUM_LABELS), f1.values[order], color=viz.BLUE, height=0.7)
axes[0].set_yticks(np.arange(NUM_LABELS), [SHORT_LABELS[i] for i in order])
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("F1 (dev)")
axes[0].set_title("Teacher: F1 per label")
axes[0].grid(axis="y", visible=False)
im = plot_confusion(teacher["dev"]["confusion"], "Teacher: confusion matrix ternormalisasi (dev)", axes[1])
fig.colorbar(im, ax=axes[1], fraction=0.04, pad=0.02, label="proporsi baris")
fig.tight_layout()
viz.savefig(fig, "fig04_teacher_per_label_confusion")
plt.show()

cm = np.asarray(teacher["dev"]["confusion"])
off = cm.copy(); np.fill_diagonal(off, 0)
pairs = sorted(((off[i, j], LABELS[i], LABELS[j]) for i in range(NUM_LABELS) for j in range(NUM_LABELS) if off[i, j]),
               reverse=True)[:6]
print("Kebingungan terbesar teacher (label benar -> prediksi, jumlah):")
for n, a, b in pairs:
    print(f"  {a} -> {b}: {n}")

### 3.4 Validasi test set LLM: kesepakatan anotator LLM dengan teacher

Anotasi LLM hanya bermanfaat bila kualitasnya setara dengan anotator yang dipakai Kuzman. Sebagai pembanding, paper Kuzman melaporkan:
- kesepakatan dua anotator manusia: Krippendorff's α = 0,728,
- kesepakatan GPT-4o dengan masing-masing manusia: setara (≈ 0,72–0,74).

Jika kesepakatan anotator LLM dengan teacher berada pada kisaran yang sama, test set LLM dapat dianggap setara kualitasnya dengan data berlabel GPT-4o yang dipakai untuk melatih teacher. Interval kepercayaan dihitung dengan bootstrap.

In [ ]:
y_llm = llm_test["label_id"].to_numpy()
y_teacher_llm = evaluation.predictions("teacher_xlmr_large", "llm_test")
agree = metrics.agreement(y_llm, y_teacher_llm)

rng = np.random.default_rng(0)
boot = {"cohen_kappa": [], "krippendorff_alpha": []}
for _ in range(1000):
    idx = rng.integers(0, len(y_llm), len(y_llm))
    a = metrics.agreement(y_llm[idx], y_teacher_llm[idx])
    for k in boot:
        boot[k].append(a[k])
agreement_table = pd.DataFrame([
    {"Pasangan": "Anotator LLM vs teacher (ini)", "Kesepakatan %": agree["percent_agreement"] * 100,
     "Cohen kappa": agree["cohen_kappa"], "Krippendorff alpha": agree["krippendorff_alpha"],
     "alpha CI95": f"{np.percentile(boot['krippendorff_alpha'], 2.5):.3f}-{np.percentile(boot['krippendorff_alpha'], 97.5):.3f}"},
    {"Pasangan": "Manusia vs manusia (Kuzman 2025)", "Kesepakatan %": np.nan, "Cohen kappa": np.nan,
     "Krippendorff alpha": 0.728, "alpha CI95": "-"},
])
save_table(agreement_table, "table3_annotation_agreement")
agreement_table

---
## 4. Pelabelan distilasi oleh teacher (*soft labels*)

Inti *knowledge distillation* (Hinton dkk., 2015): student tidak hanya belajar dari label keras (argmax), tetapi dari **seluruh distribusi probabilitas teacher**. Distribusi ini mengandung informasi tentang kemiripan antar kelas (mis. artikel tentang harga listrik: sebagian *economy*, sebagian *environment*).

Untuk setiap dokumen $d$ di korpus distilasi, teacher menghasilkan logit $\mathbf{z}^{T}(d) \in \mathbb{R}^{17}$ yang **disimpan utuh**. Distribusi target dengan temperatur $\tau$:
$$p^{T}_c(d) = \frac{\exp(z^{T}_c(d)/\tau)}{\sum_{k} \exp(z^{T}_k(d)/\tau)}$$

Korpus yang dilabeli:
| Korpus | Dokumen |
|---|---|
| EMMediaTopic (train + dev; dev hanya untuk statistik, **tidak** dipakai melatih student) | 21.000 |
| CC-News held-out (evaluasi *fidelity*) | 13.549 |
| CC-News train | 109.842 |
| CC-News train tambahan (skala data) | 156.157 |

Teacher dijalankan di GPU dalam fp16 dengan *batch* yang diurutkan berdasarkan panjang teks (mengurangi *padding*). Kecepatan di T4 ± 40–50 dokumen/detik, total ± 2 jam. Hasil disimpan per *shard* 10.000 dokumen sehingga dapat dilanjutkan jika sesi Colab terputus. Tahap ini **offline dan hanya dilakukan sekali**, sehingga penggunaan GPU di sini tidak bertentangan dengan tujuan inferensi CPU.

In [ ]:
from iptc import teacher as teacher_module

if MODE == "FULL":
    tm = teacher_module.Teacher()
    print(f"Teacher di {tm.device}: {tm.num_parameters() / 1e6:.1f} juta parameter")
    corpora = [("emmediatopic", data.load_emmediatopic()),
               ("ccnews_heldout", data.load_ccnews("heldout")),
               ("ccnews_train", data.load_ccnews("train"))]
    if RUN_DATA_SCALING:
        corpora.append(("ccnews_train_extra", data.load_ccnews("train_extra")))
    for name, df in corpora:
        t0 = time.time()
        labelled = teacher_module.pseudo_label(tm, df, name)
        print(f"{name}: {len(labelled)} dokumen, {time.time() - t0:.0f} s")
    del tm
    import gc; gc.collect(); torch.cuda.empty_cache()
    # Recompute the descriptive statistics now that teacher labels exist.
    DATA_STATS_PATH.write_text(json.dumps(data.compute_data_stats()))
    DATA_STATS = json.loads(DATA_STATS_PATH.read_text())
else:
    print("REPORT: distribusi label teacher dimuat dari", DATA_STATS_PATH)

### 4.1 Statistik label teacher

Dua pemeriksaan kewajaran sebelum distilasi:
1. **Distribusi label** teacher pada CC-News harus masuk akal, tidak runtuh ke satu kelas, dan mirip dengan distribusi berita umum.
2. **Keyakinan (confidence) teacher per bahasa.** Keyakinan yang sangat rendah di suatu bahasa menandakan teacher kurang memahami bahasa itu, sehingga labelnya kurang dapat dipercaya sebagai target distilasi.

In [ ]:
%%writefile iptc/languages.py
"""Writing-system groups of the CC-News languages (used for the per-language analysis)."""
LANG_SCRIPT = {
    **{l: "Latin" for l in ["af", "az", "bs", "ca", "cs", "cy", "da", "de", "en", "es", "et", "eu", "fi", "fr",
                            "gl", "hr", "hu", "id", "is", "it", "lt", "lv", "ms", "nl", "no", "pl", "pt", "ro",
                            "sk", "sl", "so", "sq", "sr", "sv", "sw", "tl", "tr", "vi"]},
    **{l: "Cyrillic" for l in ["be", "bg", "kk", "ky", "mk", "mn", "ru", "uk"]},
    **{l: "Arabic" for l in ["ar", "fa", "ps", "ur"]},
    **{l: "Indic" for l in ["bn", "gu", "hi", "kn", "ml", "mr", "ne", "or", "pa", "ta", "te"]},
    **{l: "CJK/Thai/Myanmar" for l in ["ja", "zh", "th", "my"]},
    **{l: "Other" for l in ["el", "he", "ka", "hy", "am", "ko"]},
}
SCRIPT_ORDER = ["Latin", "Cyrillic", "Arabic", "Indic", "CJK/Thai/Myanmar", "Other"]

In [ ]:
from iptc.languages import LANG_SCRIPT, SCRIPT_ORDER

label_dist = {}
for name, title in (("emmediatopic", "EMMediaTopic"), ("ccnews_train", "CC-News train"), ("ccnews_heldout", "CC-News held-out")):
    if name in DATA_STATS["teacher"]:
        counts = pd.Series(DATA_STATS["teacher"][name]["label_counts"]).reindex(LABELS).fillna(0)
        label_dist[title] = counts / counts.sum() * 100
label_dist = pd.DataFrame(label_dist)
label_dist.index = SHORT_LABELS
print("Distribusi label teacher (% dokumen):")
display(label_dist.round(1).sort_values("CC-News train", ascending=False))

# Figure: teacher confidence on held-out news, grouped by writing system
conf = DATA_STATS["teacher"]["ccnews_heldout"]["confidence_sample_per_lang"]
groups = {s: np.concatenate([conf[l] for l in conf if LANG_SCRIPT.get(l) == s]) for s in SCRIPT_ORDER
          if any(LANG_SCRIPT.get(l) == s for l in conf)}
fig, ax = plt.subplots(figsize=(8, 3.6))
bp = ax.boxplot(list(groups.values()), tick_labels=list(groups.keys()), showfliers=False, patch_artist=True,
                widths=0.55, medianprops={"color": viz.INK, "linewidth": 1.5})
for patch in bp["boxes"]:
    patch.set_facecolor("#cde2fb"); patch.set_edgecolor(viz.BLUE)
ax.set_ylabel("confidence teacher (max prob.)")
ax.set_title("Keyakinan teacher pada berita held-out, per sistem tulisan")
ax.grid(axis="x", visible=False)
viz.savefig(fig, "fig05_teacher_confidence_by_script")
plt.show()
mean_conf = pd.Series(DATA_STATS["teacher"]["ccnews_heldout"]["confidence_mean_per_lang"])
print("Rata-rata confidence teacher per bahasa: min", mean_conf.idxmin(), f"{mean_conf.min():.2f}",
      "| max", mean_conf.idxmax(), f"{mean_conf.max():.2f}", "| median", f"{mean_conf.median():.2f}")

---
## 5. Model student

Semua student dilatih dari **data yang sama** dan **target teacher yang sama**, sehingga perbedaannya hanya pada arsitektur dan cara belajar:

| Student | Arsitektur | Target latih | Inferensi |
|---|---|---|---|
| fastText (Joulin dkk., 2017) | bag of n-gram, embedding acak | label GPT-4o (setting Kuzman) atau argmax teacher | `fasttext` |
| **StaticKD (usulan)** | embedding statis multibahasa pra-latih → mean pooling → linear | distribusi teacher (KL) + *token dropout* | NumPy + `tokenizers` |
| MiniLM-L6 + KD | transformer 6 layer, 384 hidden (didistilasi dari XLM-R-large) | distribusi teacher (KL) | ONNX Runtime |

### 5.0 Modul data latih
`load_pool()` menggabungkan korpus distilasi (EMMediaTopic train, CC-News train, dan opsional CC-News tambahan) beserta logit teacher. **Split dev EMMediaTopic tidak pernah dimasukkan.** Untuk pemilihan model (*early stopping*), 3% data latih dipisahkan sebagai validasi internal, dengan metrik **kesepakatan dengan teacher**. Set evaluasi (dev, test LLM) tidak pernah dipakai untuk memilih hiperparameter.

In [ ]:
%%writefile iptc/training_data.py
"""Training pools for the students (the EMMediaTopic dev split is never included)."""
import numpy as np
import pandas as pd

from iptc.config import TEACHER_DIR

SOURCE_FILES = {"ccnews": "ccnews_train", "ccnews_extra": "ccnews_train_extra"}


def load_pool(sources=("emmediatopic", "ccnews")) -> pd.DataFrame:
    """Columns: text, lang, source, gpt_label_id (-1 when absent), teacher_logits."""
    frames = []
    if "emmediatopic" in sources:
        emm = pd.read_parquet(TEACHER_DIR / "emmediatopic.parquet")
        emm = emm[emm["split"] == "train"]
        frames.append(pd.DataFrame({"text": emm["text"], "lang": emm["lang"], "source": "emmediatopic",
                                    "gpt_label_id": emm["label_id"], "teacher_logits": emm["teacher_logits"]}))
    for source, file in SOURCE_FILES.items():
        if source in sources:
            cc = pd.read_parquet(TEACHER_DIR / f"{file}.parquet")
            frames.append(pd.DataFrame({"text": cc["text"], "lang": cc["lang"], "source": source,
                                        "gpt_label_id": -1, "teacher_logits": cc["teacher_logits"]}))
    return pd.concat(frames, ignore_index=True)


def teacher_probs(df: pd.DataFrame, temperature: float = 1.0) -> np.ndarray:
    logits = np.stack(df["teacher_logits"].to_numpy()) / temperature
    logits -= logits.max(axis=1, keepdims=True)
    probs = np.exp(logits)
    return probs / probs.sum(axis=1, keepdims=True)

### 5.1 Baseline fastText

fastText adalah contoh model "cepat dan ringan" yang dijadikan acuan (seperti fastText *language identification*). Kami melatih empat varian untuk memisahkan dua faktor:

| Varian | Label | Bahasa data latih | Token |
|---|---|---|---|
| `ft_gpt_word` / `ft_gpt_subword` | GPT-4o (EMMediaTopic saja = setting Kuzman) | 4 | kata / subword XLM-R |
| `ft_teacher_word` / `ft_teacher_subword` | argmax teacher | 70 | kata / subword XLM-R |
| `ft_teacher_subword_q` | argmax teacher, **dikuantisasi** (`.ftz`) | 70 | subword XLM-R |

Tokenisasi subword memakai tokenizer SentencePiece XLM-R. Tokenizer ini tetap memecah teks aksara tanpa spasi (ja/zh/th), sedangkan tokenisasi per kata tidak bisa melakukannya. fastText hanya menerima label keras, sehingga distilasi di sini berupa *hard-label distillation*. Hiperparameter: dim 100, 15 epoch (25 untuk data GPT yang kecil), lr 0,5, bigram kata, softmax.

In [ ]:
%%writefile iptc/fasttext_student.py
"""fastText students (Joulin et al., 2017)."""
import json
import re
import tempfile
from functools import lru_cache
from pathlib import Path

import numpy as np

from iptc.config import MODELS_DIR, SMOKE_TEST
from iptc.training_data import load_pool, teacher_probs

WORD_RE = re.compile(r"\w+", re.UNICODE)


def word_tokens(text: str) -> str:
    return " ".join(WORD_RE.findall(text.lower()))


@lru_cache(maxsize=None)
def _subword_tokenizer():
    from huggingface_hub import hf_hub_download
    from tokenizers import Tokenizer
    return Tokenizer.from_file(hf_hub_download("xlm-roberta-base", "tokenizer.json"))


def subword_tokens(texts: list) -> list:
    encs = _subword_tokenizer().encode_batch([t.lower() for t in texts], add_special_tokens=False)
    return [" ".join(e.tokens) for e in encs]


def tokenize(texts: list, mode: str) -> list:
    return subword_tokens(texts) if mode == "subword" else [word_tokens(t) for t in texts]


def train(name: str, labels: str = "teacher", tokens: str = "subword", dim: int = 100, epoch: int = 15,
          lr: float = 0.5, word_ngrams: int = 2, quantize: bool = False, threads: int = 4, seed: int = 0) -> Path:
    """labels: 'gpt' (EMMediaTopic GPT-4o labels only) or 'teacher' (teacher argmax, all corpora)."""
    import fasttext

    out = MODELS_DIR / "fasttext" / f"{name}.{'ftz' if quantize else 'bin'}"
    if out.exists():
        return out
    sources = ("emmediatopic",) if labels == "gpt" else ("emmediatopic", "ccnews")
    df = load_pool(sources)
    y = df["gpt_label_id"].to_numpy() if labels == "gpt" else teacher_probs(df).argmax(1)
    lines = [f"__label__{int(l)} {doc}" for l, doc in zip(y, tokenize(df["text"].tolist(), tokens))]
    np.random.default_rng(seed).shuffle(lines)
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False, encoding="utf8") as f:
        f.write("\n".join(lines) + "\n")
        train_path = f.name
    model = fasttext.train_supervised(
        input=train_path, dim=dim, epoch=epoch, lr=lr, wordNgrams=word_ngrams,
        minn=2 if tokens == "word" else 0, maxn=5 if tokens == "word" else 0,
        bucket=200_000 if SMOKE_TEST else 2_000_000, loss="softmax", thread=threads, seed=seed, verbose=0)
    if quantize:
        model.quantize(input=train_path, qnorm=True, retrain=True, cutoff=300_000, thread=threads)
    Path(train_path).unlink()
    out.parent.mkdir(parents=True, exist_ok=True)
    model.save_model(str(out))
    out.with_suffix(".json").write_text(json.dumps({"tokens": tokens, "labels": labels}))
    return out


class FastTextPredictor:
    def __init__(self, path: str):
        import fasttext

        path = Path(path)
        fasttext.FastText.eprint = lambda *args, **kwargs: None
        self.model = fasttext.load_model(str(path))
        self.tokens = json.loads(path.with_suffix(".json").read_text())["tokens"]
        tokenize(["warm-up"], self.tokens)  # load the subword tokenizer now (fair RAM measurement)
        self.size_mb = path.stat().st_size / 2**20
        self.num_parameters = None

    def predict(self, texts):
        # Low-level call: works with both fasttext builds under NumPy 2.
        out = []
        for doc in tokenize(texts, self.tokens):
            (prob, label), = self.model.f.predict(doc.replace("\n", " ") + "\n", 1, 0.0, "strict")
            out.append(int(label.removeprefix("__label__")))
        return np.array(out)

In [ ]:
from iptc import fasttext_student, training_data
from iptc.config import MODELS_DIR

FASTTEXT_RUNS = [  # name, labels, tokens, epochs, quantize
    ("ft_gpt_word", "gpt", "word", 25, False),
    ("ft_gpt_subword", "gpt", "subword", 25, False),
    ("ft_teacher_word", "teacher", "word", 15, False),
    ("ft_teacher_subword", "teacher", "subword", 15, False),
    ("ft_teacher_subword_q", "teacher", "subword", 15, True),
]
if MODE == "FULL":
    for name, labels, tokens, epochs, quantize in FASTTEXT_RUNS:
        t0 = time.time()
        path = fasttext_student.train(name, labels=labels, tokens=tokens, epoch=1 if SMOKE_TEST else epochs,
                                      quantize=quantize, threads=os.cpu_count())
        print(f"{name}: {path.stat().st_size / 2**20:.1f} MB, {time.time() - t0:.0f} s")
        evaluation.evaluate_model(f"fasttext:{path}", name)
else:
    print("REPORT: hasil fastText dimuat di Bagian 6.")

### 5.2 Metode usulan: StaticKD

#### Formulasi
Dokumen $d$ ditokenisasi menjadi token $t_1,\dots,t_n$ (tokenizer Unigram 500k milik *potion-multilingual-128M*). Setiap token punya embedding statis $\mathbf{e}_t \in \mathbb{R}^{256}$. Student menghitung:
$$\mathbf{h}(d) = \frac{1}{n}\sum_{i=1}^{n} \mathbf{e}_{t_i}, \qquad \mathbf{z}^{S}(d) = W\,\mathbf{h}(d) + \mathbf{b}, \quad W \in \mathbb{R}^{17\times 256}$$

**Inisialisasi.** $E$ diambil dari *model2vec potion-multilingual-128M* (Tulkens & van Dongen, 2024), yaitu embedding statis 101 bahasa yang didistilasi dari BGE-M3. Kata-kata bermakna sama di bahasa berbeda sudah berdekatan, sehingga pengetahuan yang dipelajari dari satu bahasa berpindah ke bahasa lain.

**Fungsi loss (knowledge distillation).** Student dilatih meniru distribusi teacher yang dilunakkan dengan temperatur $\tau = 2$:
$$\mathcal{L}_{KD} = \tau^{2}\cdot \mathrm{KL}\!\left(\mathrm{softmax}(\mathbf{z}^{T}/\tau)\;\|\;\mathrm{softmax}(\mathbf{z}^{S}/\tau)\right)$$
Faktor $\tau^2$ menjaga skala gradien (Hinton dkk., 2015). $E$, $W$, dan $\mathbf{b}$ dilatih **end-to-end**: $E$ dengan *SparseAdam* (lr $10^{-3}$, hanya baris token yang muncul di batch yang diperbarui), $W, \mathbf{b}$ dengan AdamW (lr $3\cdot10^{-3}$).

**Token dropout.** Setiap token dokumen latih dibuang dengan probabilitas $p = 0{,}2$. Ini adalah augmentasi murah yang memaksa model tidak bergantung pada beberapa kata kunci saja.

**Pelipatan tabel (untuk produksi).** Karena kepalanya linear,
$$W\Big(\frac{1}{n}\sum_i \mathbf{e}_{t_i}\Big) = \frac{1}{n}\sum_i W\mathbf{e}_{t_i} = \frac{1}{n}\sum_i \mathbf{T}_{t_i},\qquad T = E W^{\top} \in \mathbb{R}^{|V|\times 17}$$
Model produksi hanyalah **tabel 17 angka per token** (float16, ±17 MB). Inferensi = tokenisasi + rata-rata baris tabel + bias. Tidak ada perkalian matriks dan tidak ada PyTorch.

**Kuantisasi & penyimpanan.** Saat training, $E$ disimpan int8 dengan skala per baris. Tabel $T$ disimpan float16 dan akurasinya identik (lihat ablasi).

Hiperparameter final: batch 64, 6 epoch (pemilihan epoch dengan kesepakatan teacher pada validasi internal), maks. 1.024 token per dokumen.

In [ ]:
%%writefile iptc/static_student.py
"""StaticKD: static multilingual embeddings + mean pooling + linear head, trained by KD."""
import hashlib
import json
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
from tokenizers import Tokenizer

from iptc.config import LABELS, MODELS_DIR, NUM_LABELS, PROCESSED_DIR, TRAINING_HISTORY_PATH

POTION = "minishlab/potion-multilingual-128M"
MAX_TOKENS = 1024
TOKEN_CACHE_DIR = PROCESSED_DIR / "token_cache"
STATIC_DIR = MODELS_DIR / "static"


@dataclass
class StaticConfig:
    name: str
    base: str = POTION
    sources: tuple = ("emmediatopic", "ccnews")
    epochs: int = 8
    batch_size: int = 64
    lr_emb: float = 1e-3
    lr_head: float = 3e-3
    hidden: int = 0            # 0 = linear head (foldable into a token table)
    dropout: float = 0.1
    token_dropout: float = 0.0
    temperature: float = 2.0
    val_fraction: float = 0.03
    seed: int = 0


# ------------------------------------------------------------------------------ tokenisation
def load_base(name: str = POTION) -> tuple:
    from huggingface_hub import snapshot_download
    from safetensors.numpy import load_file

    path = Path(name) if Path(name).exists() else Path(snapshot_download(name))
    tokenizer = Tokenizer.from_file(str(path / "tokenizer.json"))
    embeddings = next(iter(load_file(str(path / "model.safetensors")).values())).astype(np.float32)
    return tokenizer, embeddings


def unk_id(tokenizer: Tokenizer):
    for tok in ("[UNK]", "<unk>"):
        idx = tokenizer.token_to_id(tok)
        if idx is not None:
            return idx
    return None


def encode(tokenizer: Tokenizer, texts: list, unk) -> list:
    out = []
    for e in tokenizer.encode_batch(texts, add_special_tokens=False):
        ids = np.asarray(e.ids[:MAX_TOKENS], dtype=np.int64)
        if unk is not None:
            ids = ids[ids != unk]
        out.append(ids if len(ids) else np.zeros(1, dtype=np.int64))
    return out


def encode_cached(tokenizer: Tokenizer, texts: list, unk, key: str) -> list:
    """Tokenising ~300k documents takes minutes, so token ids are cached per tokenizer + corpus."""
    digest = hashlib.md5((key + str(len(texts)) + texts[0][:200] + texts[-1][:200]).encode()).hexdigest()[:12]
    cache = TOKEN_CACHE_DIR / f"{digest}.npz"
    if cache.exists():
        d = np.load(cache)
        return np.split(d["flat"], np.cumsum(d["lengths"])[:-1])
    ids = encode(tokenizer, texts, unk)
    TOKEN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    np.savez(cache, flat=np.concatenate(ids), lengths=np.array([len(i) for i in ids]))
    return ids


# ------------------------------------------------------------------------------ training
def _append_history(name: str, history: list) -> None:
    all_hist = json.loads(TRAINING_HISTORY_PATH.read_text()) if TRAINING_HISTORY_PATH.exists() else {}
    all_hist[name] = history
    TRAINING_HISTORY_PATH.write_text(json.dumps(all_hist, indent=1))


def train(cfg: StaticConfig) -> Path:
    import torch
    import torch.nn.functional as F
    from iptc.training_data import load_pool, teacher_probs

    out = STATIC_DIR / cfg.name
    if (out / "model.npz").exists():
        return out
    torch.manual_seed(cfg.seed)
    rng = np.random.default_rng(cfg.seed)
    tokenizer, base = load_base(cfg.base)
    unk = unk_id(tokenizer)
    df = load_pool(tuple(cfg.sources))
    probs = teacher_probs(df, cfg.temperature).astype(np.float32)
    t0 = time.time()
    ids = encode_cached(tokenizer, df["text"].tolist(), unk, key=f"{cfg.base}|{list(cfg.sources)}")
    print(f"[{cfg.name}] {len(ids)} dokumen ditokenisasi ({time.time() - t0:.0f} s)")

    perm = rng.permutation(len(ids))
    n_val = max(1, int(len(ids) * cfg.val_fraction))
    val_idx, train_idx = perm[:n_val], perm[n_val:]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    emb = torch.nn.EmbeddingBag.from_pretrained(torch.from_numpy(base), freeze=False, mode="mean",
                                                sparse=True).to(device)
    layers = [torch.nn.Dropout(cfg.dropout)]
    if cfg.hidden > 0:
        layers += [torch.nn.Linear(base.shape[1], cfg.hidden), torch.nn.GELU(), torch.nn.Dropout(cfg.dropout)]
    layers.append(torch.nn.Linear(cfg.hidden or base.shape[1], NUM_LABELS))
    head = torch.nn.Sequential(*layers).to(device)
    opt_dense = torch.optim.AdamW(head.parameters(), lr=cfg.lr_head, weight_decay=1e-4)
    opt_sparse = torch.optim.SparseAdam(list(emb.parameters()), lr=cfg.lr_emb)

    def batch(idx, token_dropout=0.0):
        seqs = [ids[i] for i in idx]
        if token_dropout > 0:
            seqs = [s[rng.random(len(s)) >= token_dropout] if len(s) > 4 else s for s in seqs]
            seqs = [s if len(s) else ids[i][:1] for s, i in zip(seqs, idx)]
        offsets = np.cumsum([0] + [len(s) for s in seqs[:-1]])
        return (torch.from_numpy(np.concatenate(seqs)).to(device), torch.from_numpy(offsets).to(device))

    def val_agreement():
        emb.eval(), head.eval()
        preds = []
        with torch.no_grad():
            for s in range(0, len(val_idx), 1024):
                preds.append(head(emb(*batch(val_idx[s:s + 1024]))).argmax(-1).cpu().numpy())
        emb.train(), head.train()
        return float((np.concatenate(preds) == probs[val_idx].argmax(1)).mean())

    best, best_state, history = -1.0, None, []
    steps = max(1, len(train_idx) // cfg.batch_size)
    for epoch in range(cfg.epochs):
        rng.shuffle(train_idx)
        total = 0.0
        for step in range(steps):
            idx = train_idx[step * cfg.batch_size:(step + 1) * cfg.batch_size]
            logits = head(emb(*batch(idx, cfg.token_dropout)))
            target = torch.from_numpy(probs[idx]).to(device)
            loss = F.kl_div(F.log_softmax(logits / cfg.temperature, -1), target,
                            reduction="batchmean") * cfg.temperature ** 2
            opt_dense.zero_grad(); opt_sparse.zero_grad()
            loss.backward()
            opt_dense.step(); opt_sparse.step()
            total += loss.item()
        acc = val_agreement()
        history.append({"epoch": epoch + 1, "loss": total / steps, "val_agreement": acc})
        print(f"[{cfg.name}] epoch {epoch + 1}: loss={total / steps:.4f} val_agreement={acc:.4f}", flush=True)
        if acc > best:
            best = acc
            linears = [m for m in head if isinstance(m, torch.nn.Linear)]
            best_state = (emb.weight.detach().cpu().numpy().copy(),
                          [(m.weight.detach().cpu().numpy().copy(), m.bias.detach().cpu().numpy().copy())
                           for m in linears])
    _append_history(cfg.name, history)
    return save(cfg, tokenizer, best_state[0], best_state[1], best)


def quantize_rows(matrix: np.ndarray) -> tuple:
    scale = np.abs(matrix).max(axis=1, keepdims=True) / 127.0
    scale[scale == 0] = 1.0
    return np.round(matrix / scale).astype(np.int8), scale.astype(np.float32).ravel()


def save(cfg: StaticConfig, tokenizer, embeddings, layers, val_agreement: float) -> Path:
    out = STATIC_DIR / cfg.name
    out.mkdir(parents=True, exist_ok=True)
    q, scale = quantize_rows(embeddings)
    head = {}
    for i, (w, b) in enumerate(layers):
        head[f"W{i}"], head[f"b{i}"] = w.astype(np.float32), b.astype(np.float32)
    np.savez(out / "model.npz", emb_q=q, emb_scale=scale, remap=np.arange(len(q), dtype=np.int32), **head)
    tokenizer.save(str(out / "tokenizer.json"), pretty=False)
    meta = {**asdict(cfg), "sources": list(cfg.sources), "labels": LABELS, "val_agreement_with_teacher": val_agreement}
    (out / "meta.json").write_text(json.dumps(meta, indent=1))
    return out


# ------------------------------------------------------------------------------ compression
def export_table(model_dir) -> Path:
    """Fold a linear head into the embeddings: one float16 row of 17 logits per token."""
    src = Path(model_dir)
    dst = src.with_name(src.name + "_table")
    if (dst / "model.npz").exists():
        return dst
    d = np.load(src / "model.npz")
    if "W1" in d:
        raise ValueError("only linear-head students can be folded into a token table")
    weight = d["weight"] if "weight" in d else d["W0"]
    bias = d["bias"] if "bias" in d else d["b0"]
    emb = d["emb_q"].astype(np.float32) * d["emb_scale"][:, None]
    table = (emb[d["remap"]] @ weight.T).astype(np.float16)
    dst.mkdir(parents=True, exist_ok=True)
    np.savez(dst / "model.npz", token_table=table, bias=bias.astype(np.float32))
    Tokenizer.from_file(str(src / "tokenizer.json")).save(str(dst / "tokenizer.json"), pretty=False)
    meta = json.loads((src / "meta.json").read_text())
    meta["format"] = "token_table_fp16"
    (dst / "meta.json").write_text(json.dumps(meta, indent=1))
    return dst


def compact_tokenizer(model_dir, min_count: int = 10) -> Path:
    """Prune rare pieces from the Unigram tokenizer itself (its trie dominates RAM).

    Single characters are always kept, so every text can still be segmented.
    """
    src = Path(model_dir)
    dst = src.with_name(f"{src.name}_compact{min_count}")
    if (dst / "model.npz").exists():
        return dst
    spec = json.loads((src / "tokenizer.json").read_text(encoding="utf8"))
    vocab = spec["model"]["vocab"]
    counts = np.zeros(len(vocab), dtype=np.int64)
    largest = None
    for cache in TOKEN_CACHE_DIR.glob("*.npz"):
        flat = np.load(cache)["flat"]
        if flat.max() < len(vocab) and (largest is None or len(flat) > len(largest)):
            largest = flat
    counts += np.bincount(largest, minlength=len(vocab))
    short = np.array([len(piece.lstrip("▁")) <= 1 for piece, _ in vocab])
    keep = np.union1d(np.flatnonzero((counts >= min_count) | short), [0, 1])
    spec["model"]["vocab"] = [vocab[i] for i in keep]
    spec["model"]["unk_id"] = int(np.searchsorted(keep, spec["model"]["unk_id"]))
    spec["added_tokens"] = [t for t in spec.get("added_tokens", []) if t["id"] in (0, 1)]
    d = np.load(src / "model.npz")
    dst.mkdir(parents=True, exist_ok=True)
    np.savez(dst / "model.npz", token_table=d["token_table"][keep], bias=d["bias"])
    (dst / "tokenizer.json").write_text(json.dumps(spec, ensure_ascii=False), encoding="utf8")
    Tokenizer.from_file(str(dst / "tokenizer.json"))  # validate
    meta = json.loads((src / "meta.json").read_text())
    meta.update({"tokenizer_min_count": min_count, "vocab_size": int(len(keep))})
    (dst / "meta.json").write_text(json.dumps(meta, indent=1))
    return dst


def distill_teacher_embeddings(dims: int = 256) -> Path:
    """Ablation: static embeddings distilled from the teacher encoder itself (model2vec-style):
    each vocabulary token is encoded alone, PCA to `dims`, Zipf weighting."""
    import torch
    from safetensors.numpy import save_file
    from sklearn.decomposition import PCA
    from transformers import AutoModel, AutoTokenizer

    from iptc.config import TEACHER_MODEL_ID

    out = MODELS_DIR / "static_base" / f"teacher_distilled_{dims}"
    if (out / "model.safetensors").exists():
        return out
    tok = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID)
    model = AutoModel.from_pretrained(TEACHER_MODEL_ID).half().cuda().eval()
    vectors = np.zeros((len(tok), model.config.hidden_size), dtype=np.float32)
    with torch.inference_mode():
        for start in range(0, len(tok), 2048):
            ids = torch.arange(start, min(start + 2048, len(tok)))
            batch = torch.stack([torch.full_like(ids, tok.cls_token_id), ids,
                                 torch.full_like(ids, tok.sep_token_id)], dim=1).cuda()
            hidden = model(input_ids=batch, attention_mask=torch.ones_like(batch)).last_hidden_state
            vectors[start:start + len(ids)] = hidden[:, 1].float().cpu().numpy()
    vectors -= vectors.mean(axis=0)
    reduced = PCA(n_components=dims, random_state=0).fit_transform(vectors).astype(np.float32)
    zipf = np.log(1 + np.arange(len(reduced), dtype=np.float32))[:, None]
    zipf[:5] = 0.0
    embeddings = reduced * zipf / np.linalg.norm(reduced, axis=1).mean()
    out.mkdir(parents=True, exist_ok=True)
    save_file({"embeddings": embeddings.astype(np.float32)}, str(out / "model.safetensors"))
    tok.backend_tokenizer.save(str(out / "tokenizer.json"))
    return out


# ------------------------------------------------------------------------------ inference
def _gelu(x):
    return 0.5 * x * (1.0 + np.tanh(0.7978845608 * (x + 0.044715 * x ** 3)))


class StaticPredictor:
    """Pure NumPy inference: tokenize -> gather rows -> mean -> (MLP) -> logits."""

    def __init__(self, path):
        path = Path(path)
        d = np.load(path / "model.npz")
        self.tokenizer = Tokenizer.from_file(str(path / "tokenizer.json"))
        self.unk = unk_id(self.tokenizer)
        self.size_mb = sum(f.stat().st_size for f in path.iterdir()) / 2**20
        if "token_table" in d:  # folded deployment format
            self.table, self.scale = d["token_table"], None
            self.remap = np.arange(len(self.table), dtype=np.int32)
            self.head = [(None, d["bias"])]
            self.num_parameters = int(self.table.size + d["bias"].size)
            return
        self.remap = d["remap"]
        n = sum(1 for k in d.files if k.startswith("W"))
        layers = [(d["weight"], d["bias"])] if "weight" in d else [(d[f"W{i}"], d[f"b{i}"]) for i in range(n)]
        if len(layers) == 1:  # fold the linear head: 17 values per token
            emb = d["emb_q"].astype(np.float32) * d["emb_scale"][:, None]
            self.table, self.scale = (emb @ layers[0][0].T).astype(np.float32), None
            self.head = [(None, layers[0][1])]
        else:                 # keep int8 rows, de-quantise per document
            self.table, self.scale = d["emb_q"], d["emb_scale"]
            self.head = layers
        self.num_parameters = int(d["emb_q"].size + sum(w.size + b.size for w, b in layers))

    def logits(self, texts: list) -> np.ndarray:
        rows = [self.remap[ids] for ids in encode(self.tokenizer, texts, self.unk)]
        if self.scale is None:
            x = np.stack([self.table[r].mean(axis=0, dtype=np.float32) for r in rows])
        else:
            x = np.stack([(self.table[r] * self.scale[r, None]).mean(axis=0) for r in rows])
        for i, (w, b) in enumerate(self.head):
            x = x + b if w is None else x @ w.T + b
            if i < len(self.head) - 1:
                x = _gelu(x)
        return x

    def predict_proba(self, texts: list) -> np.ndarray:
        z = self.logits(texts)
        z = np.exp(z - z.max(axis=1, keepdims=True))
        return z / z.sum(axis=1, keepdims=True)

    def predict(self, texts):
        return self.logits(texts).argmax(axis=1)

In [ ]:
from iptc import static_student
from iptc.static_student import StaticConfig

final_sources = ("emmediatopic", "ccnews", "ccnews_extra") if RUN_DATA_SCALING else ("emmediatopic", "ccnews")
FINAL_CONFIG = StaticConfig(name="static_final", sources=final_sources, token_dropout=0.2,
                            epochs=1 if SMOKE_TEST else 6)
if MODE == "FULL":
    t0 = time.time()
    final_dir = static_student.train(FINAL_CONFIG)
    table_dir = static_student.export_table(final_dir)
    print(f"StaticKD final: {time.time() - t0:.0f} s -> {table_dir}")
    evaluation.evaluate_model(f"static:{table_dir}", "static_final_table")
FINAL_MODEL_DIR = MODELS_DIR / "static" / "static_final_table"
print(json.dumps({k: v for k, v in json.loads((FINAL_MODEL_DIR / "meta.json").read_text()).items()
                  if k not in ("labels",)}, indent=1))

### 5.3 Ablasi

Setiap varian mengubah **satu faktor** terhadap konfigurasi dasar, untuk menjawab RQ3:

| Kode | Varian | Faktor yang diuji |
|---|---|---|
| A | fastText, label GPT-4o, 4 bahasa | baseline setting Kuzman |
| B | fastText, label teacher, 70 bahasa | distilasi + cakupan bahasa |
| C | Static (potion), KD hanya EMMediaTopic (4 bahasa) | embedding yang selaras lintas bahasa saja |
| D | Static (embedding didistilasi dari teacher), KD 130k | sumber embedding |
| E | Static (potion), KD 130k | kombinasi B + C |
| F | E + token dropout 0,2 | regularisasi |
| G | E + kepala MLP 512 (+ token dropout 0,1) | kapasitas kepala |
| H | G dengan token dropout 0,2 | kapasitas + regularisasi |
| I | F + data 286k (**final**) | skala data |
| J | I dilipat ke tabel fp16 | kompresi |
| K/L | J + tokenizer dipangkas (min. frekuensi 10 / 50) | kompresi RAM |

In [ ]:
ABLATIONS = [
    StaticConfig(name="static_potion_emm_only", sources=("emmediatopic",), epochs=10),
    StaticConfig(name="static_potion_kd", epochs=10),
    StaticConfig(name="static_potion_kd_tdrop", token_dropout=0.2, epochs=8),
    StaticConfig(name="static_potion_kd_mlp", hidden=512, token_dropout=0.1, epochs=8),
    StaticConfig(name="static_potion_kd_mlp_tdrop", hidden=512, token_dropout=0.2, epochs=8),
]
if MODE == "FULL" and RUN_ABLATIONS:
    teacher_base = static_student.distill_teacher_embeddings()
    ABLATIONS.append(StaticConfig(name="static_teacherdist_kd", base=str(teacher_base), epochs=10))
    for cfg in ABLATIONS:
        if SMOKE_TEST:
            cfg.epochs = 1
        t0 = time.time()
        path = static_student.train(cfg)
        evaluation.evaluate_model(f"static:{path}", cfg.name)
        print(f"{cfg.name}: {time.time() - t0:.0f} s")
    # J: is folding the linear head into a fp16 token table lossless?
    tdrop_table = static_student.export_table(MODELS_DIR / "static" / "static_potion_kd_tdrop")
    evaluation.evaluate_model(f"static:{tdrop_table}", "static_potion_kd_tdrop_table")
    # compression variants of the final model
    for min_count in (10, 50):
        compact = static_student.compact_tokenizer(FINAL_MODEL_DIR, min_count=min_count)
        evaluation.evaluate_model(f"static:{compact}", compact.name)
else:
    print("REPORT: hasil ablasi dimuat di Bagian 6 dan 8.")

In [ ]:
# Figure: training curves (agreement with the teacher on the internal validation split)
from iptc.config import TRAINING_HISTORY_PATH
history = json.loads(TRAINING_HISTORY_PATH.read_text())
curves = [("static_potion_emm_only", "C: KD 4 bahasa (20k)", viz.GRAY),
          ("static_potion_kd", "E: KD 70 bahasa (130k)", viz.BLUE),
          ("static_potion_kd_tdrop", "F: E + token dropout", viz.AQUA),
          ("static_final", "I: final (286k)", viz.ORANGE)]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for key, label, color in curves:
    if key not in history:
        continue
    h = pd.DataFrame(history[key])
    axes[0].plot(h.epoch, h.val_agreement, marker="o", markersize=4, color=color, label=label)
    axes[1].plot(h.epoch, h.loss, marker="o", markersize=4, color=color, label=label)
axes[0].set_title("Kesepakatan dengan teacher (validasi internal)")
axes[0].set_ylabel("kesepakatan")
axes[1].set_title("Loss KD latih")
axes[1].set_ylabel(r"$\tau^2\,$KL")
for ax in axes:
    ax.set_xlabel("epoch")
axes[0].legend(loc="lower right")
fig.tight_layout()
viz.savefig(fig, "fig06_training_curves")
plt.show()
print({k: round(max(e["val_agreement"] for e in v), 4) for k, v in history.items()})

### 5.4 Student transformer kecil (pembanding "jalan tengah")

Untuk menguji apakah konteks penuh transformer sebanding dengan biayanya, kami melatih **mMiniLMv2-L6-H384** (6 layer, didistilasi dari XLM-R-large sehingga tokenizernya sama dengan teacher) dengan loss KD yang sama ($\tau=2$), 2 epoch, lr $5\cdot10^{-5}$, maks. 256 token. Model diekspor ke ONNX (fp32) dan varian kuantisasi int8 dinamis. Varian int8 dilaporkan sebagai temuan: pada model 384-dimensi ini kuantisasi dinamis menurunkan akurasi secara nyata.

In [ ]:
%%writefile iptc/transformer_student.py
"""mMiniLMv2-L6 student distilled from the teacher, exported to ONNX."""
import json
from pathlib import Path

import numpy as np

from iptc.config import LABELS, MODELS_DIR, NUM_LABELS, TRAINING_HISTORY_PATH
from iptc.training_data import load_pool, teacher_probs

BASE = "nreimers/mMiniLMv2-L6-H384-distilled-from-XLMR-Large"


def train(name: str = "minilm_l6_kd", sources=("emmediatopic", "ccnews"), epochs: int = 2, batch_size: int = 16,
          lr: float = 5e-5, max_length: int = 256, temperature: float = 2.0, val_fraction: float = 0.03,
          seed: int = 0) -> Path:
    import torch
    import torch.nn.functional as F
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

    out = MODELS_DIR / "transformer" / name
    if (out / "model.onnx").exists():
        return out
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    tokenizer = AutoTokenizer.from_pretrained(BASE)
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE, num_labels=NUM_LABELS, id2label=dict(enumerate(LABELS)),
        label2id={l: i for i, l in enumerate(LABELS)}).float().cuda()  # checkpoint is fp16: force fp32
    df = load_pool(sources)
    texts, probs = df["text"].tolist(), teacher_probs(df, temperature).astype(np.float32)
    perm = rng.permutation(len(texts))
    n_val = max(1, int(len(texts) * val_fraction))
    val_idx, train_idx = perm[:n_val], perm[n_val:]
    steps = epochs * max(1, len(train_idx) // batch_size)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.amp.GradScaler()

    def enc(idx):
        return tokenizer([texts[i] for i in idx], truncation=True, max_length=max_length, padding=True,
                         return_tensors="pt").to("cuda")

    best, history = -1.0, []
    for epoch in range(epochs):
        rng.shuffle(train_idx)
        for step in range(max(1, len(train_idx) // batch_size)):
            idx = train_idx[step * batch_size:(step + 1) * batch_size]
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**enc(idx)).logits.float()
            loss = F.kl_div(F.log_softmax(logits / temperature, -1), torch.from_numpy(probs[idx]).cuda(),
                            reduction="batchmean") * temperature ** 2
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
        model.eval()
        preds = []
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for s in range(0, len(val_idx), 128):
                preds.append(model(**enc(val_idx[s:s + 128])).logits.argmax(-1).cpu().numpy())
        model.train()
        acc = float((np.concatenate(preds) == probs[val_idx].argmax(1)).mean())
        history.append({"epoch": epoch + 1, "val_agreement": acc})
        print(f"[{name}] epoch {epoch + 1}: val_agreement={acc:.4f}", flush=True)
        if acc > best:
            best = acc
            model.save_pretrained(out)
            tokenizer.save_pretrained(out)
    all_hist = json.loads(TRAINING_HISTORY_PATH.read_text()) if TRAINING_HISTORY_PATH.exists() else {}
    all_hist[name] = history
    TRAINING_HISTORY_PATH.write_text(json.dumps(all_hist, indent=1))
    export_onnx(out)
    return out


def export_onnx(model_dir: Path) -> None:
    """model.onnx (fp32, deployed) + model_int8.onnx (dynamic int8, reference only)."""
    import torch
    from onnxruntime.quantization import QuantType, quantize_dynamic
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    model = AutoModelForSequenceClassification.from_pretrained(model_dir).float().cpu().eval()
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    dummy = tokenizer(["export"] * 2, return_tensors="pt", padding="max_length", max_length=16)
    fp32 = model_dir / "model.onnx"
    torch.onnx.export(model, (dummy["input_ids"], dummy["attention_mask"]), str(fp32),
                      input_names=["input_ids", "attention_mask"], output_names=["logits"],
                      dynamic_axes={"input_ids": {0: "batch", 1: "seq"}, "attention_mask": {0: "batch", 1: "seq"},
                                    "logits": {0: "batch"}},
                      opset_version=17, dynamo=False)
    quantize_dynamic(str(fp32), str(model_dir / "model_int8.onnx"), weight_type=QuantType.QInt8,
                     per_channel=True, op_types_to_quantize=["MatMul"])

In [ ]:
if MODE == "FULL" and RUN_TRANSFORMER_STUDENT:
    from iptc import transformer_student
    t0 = time.time()
    minilm_dir = transformer_student.train(epochs=1 if SMOKE_TEST else 2)
    print(f"MiniLM KD: {(time.time() - t0) / 60:.1f} menit")
    evaluation.evaluate_model(f"onnx:{minilm_dir}", "minilm_l6_kd_onnx")
    evaluation.evaluate_model(f"onnx-int8:{minilm_dir}", "minilm_l6_kd_onnx_int8", sets=("dev", "llm_test"))
else:
    print("REPORT / dilewati: hasil MiniLM dimuat di Bagian 6.")

---
## 6. Evaluasi akurasi terpadu

Semua model dievaluasi pada set yang sama dengan kode yang sama (`iptc/evaluation.py`). Hasil per model tersimpan di `results/accuracy/<model>.json`, berisi skor, CI95, skor per bahasa, F1 per label, confusion matrix, dan seluruh prediksi.

### 6.1 Tabel utama

In [ ]:
MODEL_NAMES = {
    "teacher_xlmr_large": "Teacher XLM-R-large (PyTorch fp32)",
    "teacher_xlmr_large_onnx_int8": "Teacher XLM-R-large (ONNX int8)",
    "minilm_l6_kd_onnx": "MiniLM-L6 + KD (ONNX fp32)",
    "minilm_l6_kd_onnx_int8": "MiniLM-L6 + KD (ONNX int8)",
    "static_final_table": "StaticKD (usulan)",
    "static_final_table_compact10": "StaticKD-compact (tokenizer 335k)",
    "static_final_table_compact50": "StaticKD-compact50 (tokenizer 222k)",
    "ft_teacher_subword_q": "fastText + KD, subword, quantized",
    "ft_teacher_subword": "fastText + KD, subword",
    "ft_teacher_word": "fastText + KD, kata",
    "ft_gpt_subword": "fastText, label GPT-4o, subword",
    "ft_gpt_word": "fastText, label GPT-4o, kata",
    "static_potion_kd_tdrop": "StaticKD 130k (tanpa data tambahan)",
}
MAIN_MODELS = ["teacher_xlmr_large", "teacher_xlmr_large_onnx_int8", "minilm_l6_kd_onnx", "minilm_l6_kd_onnx_int8",
               "static_final_table", "static_final_table_compact10", "ft_teacher_subword_q", "ft_teacher_subword",
               "ft_teacher_word", "ft_gpt_subword", "ft_gpt_word"]
available = {p.stem for p in ACCURACY_DIR.glob("*.json")}
MAIN_MODELS = [m for m in MAIN_MODELS if m in available]

def result_row(name):
    res = evaluation.load_results(name)
    row = {"Model": MODEL_NAMES.get(name, name), "key": name}
    for split, tag in (("dev", "dev"), ("llm_test", "LLM16"), ("ccnews", "agree70")):
        if split in res:
            r = res[split]
            if split == "ccnews":
                row["Kesepakatan teacher (70 bhs)"] = r["accuracy"]
            else:
                row[f"{tag} macro-F1"] = r["macro_f1"]
                row[f"{tag} CI95"] = f"{r['macro_f1_ci95'][0]:.3f}-{r['macro_f1_ci95'][1]:.3f}"
                row[f"{tag} micro-F1"] = r["micro_f1"]
    return row

main_table = pd.DataFrame([result_row(m) for m in MAIN_MODELS])
teacher_ref = main_table.set_index("key").loc["teacher_xlmr_large_onnx_int8"]
main_table["% teacher (dev)"] = main_table["dev macro-F1"] / teacher_ref["dev macro-F1"] * 100
main_table["% teacher (LLM16)"] = main_table["LLM16 macro-F1"] / teacher_ref["LLM16 macro-F1"] * 100
save_table(main_table.drop(columns=["key"]), "table4_main_accuracy")
main_table.drop(columns=["key"])

In [ ]:
# Figure: macro-F1 with bootstrap CI on the two labelled evaluation sets
plot_models = [m for m in ["teacher_xlmr_large_onnx_int8", "minilm_l6_kd_onnx", "static_final_table",
                           "ft_teacher_subword_q", "ft_gpt_subword"] if m in available]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, split, title in ((axes[0], "dev", "Dev EMMediaTopic (4 bahasa latih teacher)"),
                         (axes[1], "llm_test", "Test LLM (16 bahasa lain)")):
    vals, lo, hi, colors = [], [], [], []
    for m in plot_models:
        r = evaluation.load_results(m)[split]
        vals.append(r["macro_f1"]); lo.append(r["macro_f1"] - r["macro_f1_ci95"][0])
        hi.append(r["macro_f1_ci95"][1] - r["macro_f1"]); colors.append(viz.model_color(m))
    y = np.arange(len(plot_models))[::-1]
    ax.barh(y, vals, xerr=[lo, hi], color=colors, height=0.6, error_kw={"ecolor": viz.INK_2, "capsize": 3, "lw": 1})
    for yi, v in zip(y, vals):
        ax.text(0.01, yi, f"{v:.3f}", va="center", ha="left", fontsize=8.5, color="white", fontweight="bold")
    ax.set_xlim(0, 1)
    ax.set_title(title)
    ax.set_xlabel("macro-F1 (garis = CI95 bootstrap)")
    ax.grid(axis="y", visible=False)
axes[0].set_yticks(np.arange(len(plot_models))[::-1], [MODEL_NAMES[m] for m in plot_models])
fig.tight_layout()
viz.savefig(fig, "fig07_main_accuracy")
plt.show()

### 6.2 Uji signifikansi berpasangan

Apakah perbedaan StaticKD dengan model lain nyata atau hanya noise sampling? Semua model diuji pada dokumen yang sama, sehingga dipakai **uji berpasangan**:
- **Paired bootstrap** (2.000 resampling) untuk selisih macro-F1 $\Delta = F1_{\text{StaticKD}} - F1_{\text{model lain}}$,
- **McNemar exact** untuk akurasi,
- p-value dikoreksi **Holm–Bonferroni** per set evaluasi.

Interpretasi: $\Delta < 0$ berarti StaticKD lebih rendah. Perbedaan disebut signifikan jika $p_{\text{Holm}} < 0{,}05$.

In [ ]:
from iptc.data import load_emmediatopic
Y_TRUE = {"dev": load_emmediatopic("dev")["label_id"].to_numpy(), "llm_test": llm_test["label_id"].to_numpy()}
REFERENCE = "static_final_table"
comparisons = [m for m in ["teacher_xlmr_large_onnx_int8", "teacher_xlmr_large", "minilm_l6_kd_onnx",
                           "ft_teacher_subword_q", "ft_gpt_subword", "static_potion_kd_tdrop"] if m in available]
rows = []
for split in ("dev", "llm_test"):
    ref_pred = evaluation.predictions(REFERENCE, split)
    block = []
    for other in comparisons:
        other_pred = evaluation.predictions(other, split)
        pb = metrics.paired_bootstrap(Y_TRUE[split], ref_pred, other_pred)
        mc = metrics.mcnemar_exact(Y_TRUE[split], ref_pred, other_pred)
        block.append({"Set": split, "StaticKD vs": MODEL_NAMES.get(other, other),
                      "Δ macro-F1": pb["delta_macro_f1"], "Δ CI95": f"{pb['ci95'][0]:+.3f} s/d {pb['ci95'][1]:+.3f}",
                      "p bootstrap": pb["p_value"], "Δ akurasi": float((ref_pred == Y_TRUE[split]).mean()
                                                                        - (other_pred == Y_TRUE[split]).mean()),
                      "McNemar b/c": f"{mc['a_only_correct']}/{mc['b_only_correct']}", "p McNemar": mc["p_value"]})
    for key in ("p bootstrap", "p McNemar"):
        adjusted = metrics.holm([r[key] for r in block])
        for r, a in zip(block, adjusted):
            r[f"{key} (Holm)"] = a
    rows.extend(block)
significance = pd.DataFrame(rows)
significance["signifikan (Holm, 0.05)"] = significance["p bootstrap (Holm)"] < 0.05
save_table(significance, "table5_significance", float_format="%.4f")
significance[["Set", "StaticKD vs", "Δ macro-F1", "Δ CI95", "p bootstrap (Holm)", "Δ akurasi",
              "McNemar b/c", "p McNemar (Holm)", "signifikan (Holm, 0.05)"]]

**Interpretasi (hasil riset asli, mode REPORT).**
- StaticKD **lebih rendah secara signifikan dari teacher** (Δ macro-F1 ≈ −0,08 di dev, −0,10 di test LLM). Ini harga yang dibayar untuk percepatan ratusan kali lipat (Bagian 7).
- StaticKD **tidak berbeda signifikan dari MiniLM-L6 + KD** di kedua set (p Holm ≈ 1), padahal MiniLM ±36× lebih lambat.
- StaticKD **signifikan lebih baik dari fastText + KD di 16 bahasa lain** (Δ ≈ +0,08). Di dev (4 bahasa yang datanya banyak) selisihnya kecil dan tidak signifikan. Keunggulan StaticKD terutama ada pada generalisasi lintas bahasa.
- Menggandakan data distilasi (130k → 286k) tidak memberi perbedaan signifikan (Δ ≈ +0,005), sehingga arsitektur bag-of-tokens sudah jenuh kapasitasnya.

### 6.3 Tabel ablasi dan kurva skala data (RQ3)

In [ ]:
ABLATION_ROWS = [
    ("A", "fastText, label GPT-4o, 4 bahasa (setting Kuzman)", "ft_gpt_subword"),
    ("B", "fastText, label teacher, 70 bahasa", "ft_teacher_subword"),
    ("C", "Static potion, KD 4 bahasa (20k)", "static_potion_emm_only"),
    ("D", "Static dari embedding teacher, KD 70 bahasa (130k)", "static_teacherdist_kd"),
    ("E", "Static potion, KD 70 bahasa (130k)", "static_potion_kd"),
    ("F", "E + token dropout 0,2", "static_potion_kd_tdrop"),
    ("G", "E + kepala MLP 512 (+ token dropout 0,1)", "static_potion_kd_mlp"),
    ("H", "G dengan token dropout 0,2", "static_potion_kd_mlp_tdrop"),
    ("J", "F dilipat ke tabel 17 logit/token (fp16)", "static_potion_kd_tdrop_table"),
    ("I", "F + data 286k (FINAL, format tabel)", "static_final_table"),
    ("K", "I + tokenizer dipangkas (min. frekuensi 10)", "static_final_table_compact10"),
    ("L", "I + tokenizer dipangkas (min. frekuensi 50)", "static_final_table_compact50"),
]
rows = []
for code_, desc, key in ABLATION_ROWS:
    if key not in available:
        continue
    res = evaluation.load_results(key)
    rows.append({"Kode": code_, "Varian": desc, "dev macro-F1": res["dev"]["macro_f1"],
                 "LLM16 macro-F1": res.get("llm_test", {}).get("macro_f1", np.nan),
                 "kesepakatan teacher 70 bhs": res.get("ccnews", {}).get("accuracy", np.nan), "key": key})
ablation = pd.DataFrame(rows)
save_table(ablation.drop(columns=["key"]), "table7_ablation")
ablation.drop(columns=["key"])

In [ ]:
# Figure: ablation (dev vs LLM16 macro-F1) and data scaling
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.6, 1]})
ab = ablation[ablation["Kode"].isin(list("ABCDEFGHI"))]
y = np.arange(len(ab))[::-1]
axes[0].barh(y + 0.19, ab["dev macro-F1"], height=0.36, color=viz.BLUE, label="dev (4 bahasa)")
axes[0].barh(y - 0.19, ab["LLM16 macro-F1"], height=0.36, color=viz.ORANGE, label="test LLM (16 bahasa lain)")
import textwrap
axes[0].set_yticks(y, [textwrap.fill(f"{c}  {d}", 40) for c, d in zip(ab["Kode"], ab["Varian"])], fontsize=7.5)
axes[0].set_xlim(0, 1.08)
axes[0].set_xlabel("macro-F1")
axes[0].set_title("Ablasi komponen StaticKD")
axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
axes[0].grid(axis="y", visible=False)

scale_points = {"tanpa token dropout": [("static_potion_emm_only", 20_000), ("static_potion_kd", 129_842)],
                "dengan token dropout 0,2": [("static_potion_kd_tdrop", 129_842), ("static_final_table", 285_999)]}
for (label, pts), color in zip(scale_points.items(), (viz.GRAY, viz.ORANGE)):
    pts = [(k, n) for k, n in pts if k in available]
    if not pts:
        continue
    xs = [n for _, n in pts]
    for split, ls in (("dev", "-"), ("llm_test", "--")):
        ys = [evaluation.load_results(k)[split]["macro_f1"] for k, _ in pts]
        axes[1].plot(xs, ys, ls, marker="o", markersize=6, color=color,
                     label=f"{label} — {'dev' if split == 'dev' else 'LLM16'}")
axes[1].set_xscale("log")
axes[1].set_xticks([20_000, 50_000, 100_000, 200_000, 300_000], ["20k", "50k", "100k", "200k", "300k"])
axes[1].xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
axes[1].set_xlabel("dokumen distilasi (skala log)")
axes[1].set_ylabel("macro-F1")
axes[1].set_title("Skala data distilasi")
axes[1].legend(fontsize=7.5, loc="lower right")
fig.tight_layout()
viz.savefig(fig, "fig08_ablation_scaling")
plt.show()

---
## 7. Efisiensi CPU (RQ1, RQ4)

Protokol benchmark (dirancang untuk mensimulasikan **server kecil tanpa GPU**):
- GPU disembunyikan (`CUDA_VISIBLE_DEVICES=""`), jumlah thread dikunci (OMP/MKL/ONNX Runtime/`tokenizers`) dan proses dipin ke core fisik tertentu (`cpu_affinity`).
- **Latensi** = 1 dokumen per panggilan (skenario streaming): mean, p50, p95.
- **Throughput** = dokumen/detik dengan batch (64 untuk student ringan, 8 untuk transformer).
- **RAM model** = kenaikan RSS proses setelah model dimuat (di luar interpreter Python).
- Setiap konfigurasi berjalan di **proses baru**.
- Sampel teks: dokumen dev (median ±220 kata), sama untuk semua model.

> **Catatan metodologi penting (CPU hybrid).** Laptop pengukuran (Intel Core Ultra 7 155H) memiliki P-core, E-core (±2,5× lebih lambat), dan LP E-core (±3,7× lebih lambat). Tanpa pinning, Windows memindahkan thread ke core lambat dan hasil multi-thread menjadi tidak stabil (bahkan lebih lambat dari 1 thread). Karena itu setiap proses dipin ke P-core fisik (logical CPU 0, 10, 12, 14). Di Colab (vCPU seragam) pinning memakai CPU pertama yang tersedia.

In [ ]:
%%writefile iptc/benchmark.py
"""CPU-only latency / throughput / RAM benchmark of one model, in a fresh process.

    python -m iptc.benchmark --model static:<dir> --name StaticKD --threads 1 --out results/x.jsonl
"""
import argparse
import os
import sys

os.environ["CUDA_VISIBLE_DEVICES"] = ""  # CPU only


def _limit_threads(threads: int) -> list:
    for var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS", "RAYON_NUM_THREADS"):
        os.environ[var] = str(threads)
    os.environ["TOKENIZERS_PARALLELISM"] = "true" if threads > 1 else "false"
    import psutil
    cpus = [int(c) for c in os.environ["BENCH_CPUS"].split(",")] if os.environ.get("BENCH_CPUS") \
        else sorted(psutil.Process().cpu_affinity())
    cpus = cpus[:threads]
    psutil.Process().cpu_affinity(cpus)
    return cpus


def main() -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True)
    p.add_argument("--name", required=True)
    p.add_argument("--threads", type=int, default=1)
    p.add_argument("--n-latency", type=int, default=100)
    p.add_argument("--n-throughput", type=int, default=500)
    p.add_argument("--batch-size", type=int, default=32)
    p.add_argument("--out", required=True)
    args = p.parse_args()
    cpus = _limit_threads(args.threads)

    import json, platform, time
    import numpy as np
    import psutil
    from iptc.data import load_emmediatopic
    from iptc.predictors import load_predictor

    rss = lambda: psutil.Process().memory_info().rss / 2**20
    texts = load_emmediatopic("dev").sample(frac=1.0, random_state=0)["text"].tolist()
    texts = (texts * 20)[:max(args.n_latency, args.n_throughput)]
    rss_before = rss()
    t0 = time.perf_counter()
    model = load_predictor(args.model, args.threads)
    load_s = time.perf_counter() - t0
    rss_model = rss() - rss_before

    for t in texts[:3]:
        model.predict([t])
    lat = []
    for t in texts[:args.n_latency]:
        s = time.perf_counter(); model.predict([t]); lat.append((time.perf_counter() - s) * 1000)
    model.predict(texts[:args.batch_size])
    s = time.perf_counter()
    for i in range(0, args.n_throughput, args.batch_size):
        model.predict(texts[i:i + args.batch_size])
    dps = args.n_throughput / (time.perf_counter() - s)

    lat = np.array(lat)
    result = {"model": args.name, "spec": args.model, "threads": args.threads, "pinned_cpus": cpus,
              "hardware": platform.processor() or platform.machine(), "size_mb": round(model.size_mb, 2),
              "num_parameters": model.num_parameters, "load_seconds": round(load_s, 2),
              "rss_baseline_mb": round(rss_before, 1), "rss_model_mb": round(rss_model, 1),
              "rss_peak_mb": round(rss(), 1), "latency_ms_mean": round(float(lat.mean()), 3),
              "latency_ms_p50": round(float(np.percentile(lat, 50)), 3),
              "latency_ms_p95": round(float(np.percentile(lat, 95)), 3),
              "docs_per_sec": round(dps, 2), "batch_size": args.batch_size}
    print(json.dumps(result))
    with open(args.out, "a", encoding="utf8") as f:
        f.write(json.dumps(result) + "\n")


if __name__ == "__main__":
    main()

In [ ]:
BENCH_LAPTOP = RESULTS_DIR / "cpu_benchmark.jsonl"          # Core Ultra 7 155H, P-core pinned (paper)
BENCH_COLAB = RESULTS_DIR / "cpu_benchmark_colab.jsonl"     # produced by FULL mode in Colab

BENCH_MODELS = [  # name, spec, heavy
    ("static_final_table", f"static:{MODELS_DIR / 'static' / 'static_final_table'}", False),
    ("static_final_table_compact10", f"static:{MODELS_DIR / 'static' / 'static_final_table_compact10'}", False),
    ("ft_teacher_subword_q", f"fasttext:{MODELS_DIR / 'fasttext' / 'ft_teacher_subword_q.ftz'}", False),
    ("minilm_l6_kd_onnx", f"onnx:{MODELS_DIR / 'transformer' / 'minilm_l6_kd'}", True),
    ("teacher_xlmr_large_onnx_int8", "teacher-onnx-int8", True),
    ("teacher_xlmr_large", "teacher-pytorch", True),
]

def run_benchmark(name, spec, heavy, threads, out):
    cmd = [sys.executable, "-m", "iptc.benchmark", "--model", spec, "--name", name, "--threads", str(threads),
           "--out", str(out), "--n-latency", "20" if heavy else "300", "--n-throughput", "40" if heavy else "2000",
           "--batch-size", "8" if heavy else "64"]
    if SMOKE_TEST:
        cmd[cmd.index("--n-latency") + 1], cmd[cmd.index("--n-throughput") + 1] = "5", "16"
    env = {**os.environ, "PYTHONPATH": LIB_DIR}
    proc = subprocess.run(cmd, cwd=LIB_DIR, env=env, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"[{name}] gagal:\n{proc.stderr[-1500:]}")

if MODE == "FULL":
    BENCH_COLAB.unlink(missing_ok=True)
    thread_counts = sorted({1, min(4, os.cpu_count())})
    for name, spec, heavy in BENCH_MODELS:
        kind, _, path = spec.partition(":")
        if path and not Path(path).exists():
            print("lewati (model tidak ada):", name); continue
        for t in thread_counts:
            t0 = time.time()
            run_benchmark(name, spec, heavy, t, BENCH_COLAB)
            print(f"{name} @ {t} thread: {time.time() - t0:.0f} s")

bench_file = BENCH_COLAB if (MODE == "FULL" and BENCH_COLAB.exists()) else BENCH_LAPTOP
bench = pd.read_json(bench_file, lines=True).drop_duplicates(["model", "threads"], keep="last")
print("Sumber benchmark:", bench_file.name)

In [ ]:
eff_rows = []
for m in [b[0] for b in BENCH_MODELS]:
    sub = bench[bench.model == m].set_index("threads")
    if sub.empty:
        continue
    one = sub.loc[1]
    multi_t = max(sub.index)
    multi = sub.loc[multi_t]
    acc = evaluation.load_results(m) if m in available else None
    eff_rows.append({
        "Model": MODEL_NAMES.get(m, m), "key": m,
        "dev macro-F1": acc["dev"]["macro_f1"] if acc else np.nan,
        "LLM16 macro-F1": acc["llm_test"]["macro_f1"] if acc else np.nan,
        "Latensi p50 1c (ms)": one.latency_ms_p50, "Latensi p95 1c (ms)": one.latency_ms_p95,
        "Throughput 1c (dok/s)": one.docs_per_sec, f"Throughput {multi_t}c (dok/s)": multi.docs_per_sec,
        "RAM model (MB)": one.rss_model_mb, "Ukuran file (MB)": one.size_mb,
    })
efficiency = pd.DataFrame(eff_rows)
ref = efficiency.set_index("key")
if "teacher_xlmr_large_onnx_int8" in ref.index and "static_final_table" in ref.index:
    speedup = ref.loc["static_final_table", "Throughput 1c (dok/s)"] / ref.loc["teacher_xlmr_large_onnx_int8", "Throughput 1c (dok/s)"]
    speedup_pt = ref.loc["static_final_table", "Throughput 1c (dok/s)"] / ref.loc["teacher_xlmr_large", "Throughput 1c (dok/s)"]
    print(f"StaticKD vs teacher ONNX int8: {speedup:.0f}x lebih cepat | vs teacher PyTorch: {speedup_pt:.0f}x (1 core)")
    docs_per_day = 1_000_000
    for key in ("teacher_xlmr_large", "teacher_xlmr_large_onnx_int8", "static_final_table"):
        cores = docs_per_day / 86400 / ref.loc[key, "Throughput 1c (dok/s)"]
        print(f"  core yang dibutuhkan untuk 1 juta artikel/hari — {MODEL_NAMES[key]}: {cores:.2f}")
save_table(efficiency.drop(columns=["key"]), "table6_efficiency", float_format="%.2f")
efficiency.drop(columns=["key"])

In [ ]:
# Figure: the accuracy-efficiency trade-off (log scale throughput) — the central figure
fig, ax = plt.subplots(figsize=(8.2, 4.8))
for _, r in efficiency.iterrows():
    color = viz.model_color(r["key"])
    ax.scatter(r["Throughput 1c (dok/s)"], r["LLM16 macro-F1"], s=90, color=color, edgecolor="white",
               linewidth=1.5, zorder=3)
    offset = {"teacher_xlmr_large_onnx_int8": (8, 5), "teacher_xlmr_large": (8, -12),
              "static_final_table": (-10, -14), "static_final_table_compact10": (8, -10)}.get(r["key"], (8, -3))
    ax.annotate(r["Model"], (r["Throughput 1c (dok/s)"], r["LLM16 macro-F1"]), textcoords="offset points",
                xytext=offset, fontsize=8, color=viz.INK_2, ha="right" if offset[0] < 0 else "left")
ax.set_xscale("log")
ax.set_xlabel("throughput 1 core CPU (dokumen/detik, skala log)")
ax.set_ylabel("macro-F1 test LLM (16 bahasa lain)")
ax.set_title("Trade-off akurasi–efisiensi di CPU")
ax.set_xlim(0.1, 5000)
ax.set_ylim(0.55, 0.9)
viz.savefig(fig, "fig09_accuracy_vs_throughput")
plt.show()

# Figure: latency, RAM and file size (small multiples, one axis each)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
eff = efficiency.iloc[::-1]
y = np.arange(len(eff))
colors = [viz.model_color(k) for k in eff["key"]]
for ax, col, title in ((axes[0], "Latensi p50 1c (ms)", "Latensi per artikel, 1 core (ms, log)"),
                       (axes[1], "RAM model (MB)", "RAM model (MB, log)"),
                       (axes[2], "Ukuran file (MB)", "Ukuran file (MB, log)")):
    ax.barh(y, eff[col], color=colors, height=0.6)
    ax.set_xscale("log")
    ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
    ax.set_title(title)
    ax.grid(axis="y", visible=False)
    for yi, v in zip(y, eff[col]):
        ax.text(v * 1.08, yi, f"{v:,.1f}", va="center", fontsize=7.5, color=viz.INK_2)
axes[0].set_yticks(y, eff["Model"])
fig.tight_layout()
viz.savefig(fig, "fig10_latency_ram_size")
plt.show()

---
## 8. Analisis

### 8.1 Apakah bahasa tertentu cenderung lebih lemah? (RQ4)

Kinerja per bahasa diukur sebagai **kesepakatan StaticKD dengan teacher pada berita held-out** (70 bahasa, situs tidak terlihat). Kami menguji tiga hipotesis:
1. **Jumlah data latih**: bahasa dengan data distilasi sedikit lebih lemah. Diuji dengan korelasi Spearman antara kesepakatan dan log jumlah dokumen latih.
2. **Keragaman sumber**: jumlah situs berita di data latih. Diuji dengan Spearman.
3. **Sistem tulisan** (Latin, Sirilik, Arab, Indik, CJK/Thai/Myanmar, lainnya). Diuji dengan Kruskal–Wallis, dan dengan membandingkan $R^2$ regresi *kesepakatan ~ log(data)* dengan dan tanpa dummy sistem tulisan. Jika sistem tulisan tidak menambah daya jelas setelah jumlah data dikontrol, kelemahan lebih disebabkan kurangnya data daripada aksara.

Bahasa dengan < 20 dokumen held-out dikecualikan dari uji statistik karena estimasinya terlalu bising (tetap ditampilkan di tabel).

In [ ]:
from scipy import stats as sstats
final_res = evaluation.load_results("static_final_table")
ft_res = evaluation.load_results("ft_teacher_subword_q") if "ft_teacher_subword_q" in available else None
emm_train_per_lang = emm_counts.query("split == 'train'").groupby("lang").n.sum()
lang_rows = []
for lang, r in final_res["ccnews"]["per_language"].items():
    stats_lang = DATA_STATS["ccnews_per_lang"].get(lang, {})
    n_train = stats_lang.get("train", 0) + stats_lang.get("train_extra", 0) + int(emm_train_per_lang.get(lang, 0))
    lang_rows.append({"bahasa": lang, "aksara": LANG_SCRIPT.get(lang, "Other"), "kesepakatan StaticKD": r["accuracy"],
                      "kesepakatan fastText": ft_res["ccnews"]["per_language"][lang]["accuracy"] if ft_res else np.nan,
                      "n held-out": r["n"], "dok latih": n_train, "situs latih": stats_lang.get("train_domains", 0),
                      "confidence teacher": DATA_STATS["teacher"]["ccnews_heldout"]["confidence_mean_per_lang"].get(lang, np.nan)})
per_lang = pd.DataFrame(lang_rows).sort_values("kesepakatan StaticKD", ascending=False)
reliable = per_lang[per_lang["n held-out"] >= 20].copy()
reliable["log10 dok latih"] = np.log10(reliable["dok latih"].clip(lower=1))

rho_data, p_data = sstats.spearmanr(reliable["log10 dok latih"], reliable["kesepakatan StaticKD"])
rho_dom, p_dom = sstats.spearmanr(reliable["situs latih"], reliable["kesepakatan StaticKD"])
rho_conf, p_conf = sstats.spearmanr(reliable["confidence teacher"], reliable["kesepakatan StaticKD"])
groups = [g["kesepakatan StaticKD"].values for _, g in reliable.groupby("aksara") if len(g) >= 2]
h_script, p_script = sstats.kruskal(*groups) if len(groups) >= 2 else (np.nan, np.nan)

def r_squared(X, y):
    X = np.column_stack([np.ones(len(y)), X])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    return 1 - resid.var() / y.var()

y_agree = reliable["kesepakatan StaticKD"].to_numpy()
dummies = pd.get_dummies(reliable["aksara"], drop_first=True).to_numpy(dtype=float)
r2_data = r_squared(reliable[["log10 dok latih"]].to_numpy(), y_agree)
r2_script = r_squared(dummies, y_agree)
r2_both = r_squared(np.column_stack([reliable[["log10 dok latih"]].to_numpy(), dummies]), y_agree)

lang_stats = pd.DataFrame([
    {"Uji": "Spearman: kesepakatan vs log(dok latih)", "statistik": rho_data, "p": p_data},
    {"Uji": "Spearman: kesepakatan vs jumlah situs latih", "statistik": rho_dom, "p": p_dom},
    {"Uji": "Spearman: kesepakatan vs confidence teacher", "statistik": rho_conf, "p": p_conf},
    {"Uji": "Kruskal-Wallis: kesepakatan antar sistem tulisan (H)", "statistik": h_script, "p": p_script},
    {"Uji": "R² regresi: log(data) saja", "statistik": r2_data, "p": np.nan},
    {"Uji": "R² regresi: sistem tulisan saja", "statistik": r2_script, "p": np.nan},
    {"Uji": "R² regresi: log(data) + sistem tulisan", "statistik": r2_both, "p": np.nan},
])
print(f"Bahasa yang diuji (n held-out >= 20): {len(reliable)} dari {len(per_lang)}")
save_table(lang_stats, "table8_language_statistics", float_format="%.4f")
save_table(per_lang, "tableA_per_language")
display(lang_stats)
display(reliable.groupby("aksara")["kesepakatan StaticKD"].agg(["count", "mean", "median", "min", "max"]).sort_values("mean"))

In [ ]:
# Figure: per-language agreement vs training data (left) and StaticKD vs fastText per language (right)
script_markers = {"Latin": "o", "Cyrillic": "s", "Arabic": "^", "Indic": "D", "CJK/Thai/Myanmar": "P", "Other": "X"}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for script, g in per_lang.groupby("aksara"):
    axes[0].scatter(g["dok latih"].clip(lower=1), g["kesepakatan StaticKD"], marker=script_markers.get(script, "o"),
                    s=np.clip(g["n held-out"], 10, 300) / 3 + 10, color=viz.ORANGE, alpha=0.75,
                    edgecolor="white", linewidth=0.8, label=script)
for _, r in per_lang.iterrows():
    if r["kesepakatan StaticKD"] < 0.65 or (r["dok latih"] < 100 and r["kesepakatan StaticKD"] < 0.75):
        axes[0].annotate(r["bahasa"], (max(r["dok latih"], 1), r["kesepakatan StaticKD"]), textcoords="offset points",
                         xytext={"ur": (5, -10), "ms": (-12, 3)}.get(r["bahasa"], (4, 3)), fontsize=7.5, color=viz.INK_2)
axes[0].set_xscale("log")
axes[0].set_xlabel("dokumen latih distilasi per bahasa (log; ukuran titik = n held-out)")
axes[0].set_ylabel("kesepakatan StaticKD dengan teacher")
axes[0].set_title(f"Per bahasa: data vs kinerja (Spearman ρ = {rho_data:.2f}, p = {p_data:.1e})")
axes[0].legend(title="sistem tulisan (marker)", fontsize=7.5, title_fontsize=8, loc="lower right")

if ft_res:
    axes[1].plot([0, 1], [0, 1], color=viz.AXIS, linewidth=1, zorder=1)
    axes[1].scatter(per_lang["kesepakatan fastText"], per_lang["kesepakatan StaticKD"], s=28, color=viz.ORANGE,
                    edgecolor="white", linewidth=0.8, zorder=3)
    gap = per_lang.assign(d=per_lang["kesepakatan StaticKD"] - per_lang["kesepakatan fastText"]).sort_values("d")
    for _, r in pd.concat([gap.tail(6), gap.head(2)]).iterrows():
        axes[1].annotate(r["bahasa"], (r["kesepakatan fastText"], r["kesepakatan StaticKD"]), textcoords="offset points",
                         xytext=(4, -8), fontsize=7.5, color=viz.INK_2)
    axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1)
    axes[1].set_xlabel("kesepakatan fastText + KD")
    axes[1].set_ylabel("kesepakatan StaticKD")
    better = (per_lang["kesepakatan StaticKD"] > per_lang["kesepakatan fastText"]).mean() * 100
    axes[1].set_title(f"StaticKD vs fastText per bahasa ({better:.0f}% bahasa di atas garis)")
fig.tight_layout()
viz.savefig(fig, "fig11_per_language")
plt.show()
print("5 bahasa terlemah:", per_lang.tail(5)[["bahasa", "kesepakatan StaticKD", "dok latih", "n held-out"]].to_dict("records"))

**Interpretasi (hasil riset asli, mode REPORT): kecenderungan per bahasa.**
- **Jumlah data adalah faktor utama yang teramati.** Kesepakatan berkorelasi positif dengan log jumlah dokumen latih (ρ ≈ 0,39, p ≈ 0,002) dan dengan jumlah situs sumber (ρ ≈ 0,35). Bahasa terlemah (ps, ne, am, ur, ms) semuanya punya < 50 dokumen latih.
- **Sistem tulisan tidak signifikan** (Kruskal–Wallis p ≈ 0,12). Bahasa beraksara Arab, Sirilik, Indik, dan CJK rata-rata tidak lebih buruk dari Latin bila datanya cukup (mis. ar 0,86, ru 0,87, hi 0,80, zh 0,82). Ini berkat embedding potion yang selaras lintas bahasa. Sebaliknya, fastText jatuh di aksara non-Latin (be, pa, zh, ka).
- **Keyakinan teacher adalah prediktor terkuat** (ρ ≈ 0,66): di bahasa tempat teacher sendiri ragu, student juga lebih sering berbeda. Sebagian "kelemahan" adalah ketidakpastian teacher yang ikut tertiru.
- **Anomali: bahasa Turki** di test LLM (0,25 vs teacher 0,65–0,75) walaupun datanya cukup (±3.400 dokumen) dan kesepakatan di held-out 0,74. Kemungkinan akibat sumber berita Turki yang homogen (sedikit situs) dan hanya 20 dokumen uji. Kasus ini perlu diselidiki lebih lanjut.
- Implikasi praktis: untuk bahasa target yang lemah, solusi termurah adalah **menambah berita bahasa tersebut lalu melabelinya dengan teacher**. Arsitektur tidak perlu diubah.

In [ ]:
# Test LLM per language: teacher vs StaticKD vs fastText (20 docs per language -> wide uncertainty)
rows = []
for lang in sorted(llm_test.lang.unique()):
    row = {"bahasa": lang, "aksara": LANG_SCRIPT.get(lang)}
    for key, label in (("teacher_xlmr_large_onnx_int8", "teacher"), ("static_final_table", "StaticKD"),
                       ("ft_teacher_subword_q", "fastText"), ("minilm_l6_kd_onnx", "MiniLM")):
        if key in available:
            row[label] = evaluation.load_results(key)["llm_test"]["per_language"][lang]["accuracy"]
    rows.append(row)
llm_lang = pd.DataFrame(rows).set_index("bahasa")
llm_lang["StaticKD - teacher"] = llm_lang["StaticKD"] - llm_lang["teacher"]
save_table(llm_lang.reset_index(), "tableB_llm_test_per_language")
llm_lang.sort_values("StaticKD - teacher")

### 8.2 Analisis per label

In [ ]:
label_models = [m for m in ["teacher_xlmr_large_onnx_int8", "static_final_table", "ft_teacher_subword_q"] if m in available]
fig, axes = plt.subplots(1, 2, figsize=(13, 5.6), gridspec_kw={"width_ratios": [1, 1.35]})
order = np.argsort(pd.Series(evaluation.load_results("static_final_table")["dev"]["per_label_f1"]).reindex(LABELS).values)
y = np.arange(NUM_LABELS)
height = 0.8 / len(label_models)
for i, m in enumerate(label_models):
    f1 = pd.Series(evaluation.load_results(m)["dev"]["per_label_f1"]).reindex(LABELS).values[order]
    axes[0].barh(y + (len(label_models) / 2 - i - 0.5) * height, f1, height=height * 0.92, color=viz.model_color(m),
                 label=MODEL_NAMES[m])
axes[0].set_yticks(y, [SHORT_LABELS[i] for i in order])
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("F1 (dev)")
axes[0].set_title("F1 per label")
axes[0].legend(loc="upper center", bbox_to_anchor=(0.45, -0.1), fontsize=7.5, ncol=1)
axes[0].grid(axis="y", visible=False)
im = plot_confusion(evaluation.load_results("static_final_table")["dev"]["confusion"],
                    "StaticKD: confusion matrix ternormalisasi (dev)", axes[1])
fig.colorbar(im, ax=axes[1], fraction=0.04, pad=0.02, label="proporsi baris")
fig.tight_layout()
viz.savefig(fig, "fig12_per_label")
plt.show()

label_table = pd.DataFrame({MODEL_NAMES[m]: pd.Series(evaluation.load_results(m)["dev"]["per_label_f1"]) for m in label_models})
label_table["gap StaticKD - teacher"] = label_table[MODEL_NAMES["static_final_table"]] - label_table[MODEL_NAMES["teacher_xlmr_large_onnx_int8"]]
save_table(label_table.reset_index(names="label"), "tableC_per_label_f1")
label_table.sort_values("gap StaticKD - teacher")

### 8.3 Kalibrasi probabilitas StaticKD

Probabilitas softmax StaticKD dipakai untuk mode cascade dan kandidat multi-label, sehingga perlu diperiksa kalibrasinya: apakah prediksi dengan confidence 0,9 memang benar ±90% dari waktu? Kalibrasi diukur dengan *reliability diagram* dan ECE. Inferensi StaticKD hanya butuh beberapa detik di CPU, sehingga bagian ini selalu dijalankan (juga pada mode REPORT).

In [ ]:
from iptc.static_student import StaticPredictor
student = StaticPredictor(FINAL_MODEL_DIR)
EVAL_TEXTS = {"dev": load_emmediatopic("dev")["text"].tolist(), "llm_test": llm_test["text"].tolist()}
PROBS = {split: np.concatenate([student.predict_proba(t[i:i + 64]) for i in range(0, len(t), 64)])
         for split, t in EVAL_TEXTS.items()}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
calib_rows = []
for ax, split, title in ((axes[0], "dev", "Dev (4 bahasa)"), (axes[1], "llm_test", "Test LLM (16 bahasa)")):
    conf = PROBS[split].max(1)
    correct = PROBS[split].argmax(1) == Y_TRUE[split]
    ece, bins = metrics.expected_calibration_error(conf, correct)
    b = pd.DataFrame(bins)
    ax.plot([0, 1], [0, 1], color=viz.AXIS, linewidth=1)
    ax.bar(b.confidence, b.accuracy, width=1 / 15 * 0.9, color="#cde2fb", edgecolor=viz.BLUE, linewidth=1)
    ax.plot(b.confidence, b.accuracy, marker="o", markersize=4, color=viz.BLUE)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("confidence rata-rata bin")
    ax.set_ylabel("akurasi bin")
    ax.set_title(f"{title}: ECE = {ece:.3f}")
    calib_rows.append({"set": split, "ECE": ece, "confidence rata-rata": conf.mean(), "akurasi": correct.mean()})
fig.suptitle("Reliability diagram StaticKD", fontweight="bold", x=0.01, ha="left")
fig.tight_layout()
viz.savefig(fig, "fig13_calibration")
plt.show()
pd.DataFrame(calib_rows)

### 8.4 Mode cascade: StaticKD → teacher

Jika akurasi maksimum diperlukan, StaticKD dapat bertindak sebagai **penyaring**. Artikel dengan confidence ≥ ambang $\theta$ diputuskan StaticKD, sisanya dikirim ke teacher. Kurva berikut menunjukkan porsi trafik yang ditangani student dan akurasi gabungan, serta estimasi throughput efektif:
$$\text{waktu per dokumen} \approx t_{\text{student}} + (1-\text{coverage}(\theta))\cdot t_{\text{teacher}}$$

In [ ]:
thresholds = np.round(np.arange(0.0, 0.96, 0.05), 2)
t_student = 1 / bench.query("model == 'static_final_table' and threads == 1").docs_per_sec.iloc[0]
t_teacher = 1 / bench.query("model == 'teacher_xlmr_large_onnx_int8' and threads == 1").docs_per_sec.iloc[0]
cascade_rows = []
for split in ("dev", "llm_test"):
    teacher_pred = evaluation.predictions("teacher_xlmr_large_onnx_int8", split)
    conf, pred = PROBS[split].max(1), PROBS[split].argmax(1)
    for th in thresholds:
        keep = conf >= th
        combined = np.where(keep, pred, teacher_pred)
        cascade_rows.append({"set": split, "ambang": th, "coverage student": keep.mean(),
                             "akurasi pada porsi student": (pred[keep] == Y_TRUE[split][keep]).mean() if keep.any() else np.nan,
                             "cascade macro-F1": metrics.fast_macro_f1(Y_TRUE[split], combined),
                             "cascade micro-F1": (combined == Y_TRUE[split]).mean(),
                             "throughput 1c (dok/s)": 1 / (t_student + (1 - keep.mean()) * t_teacher)})
cascade = pd.DataFrame(cascade_rows)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.9))
for split, color in (("dev", viz.BLUE), ("llm_test", viz.ORANGE)):
    c = cascade[cascade.set == split]
    label = "dev" if split == "dev" else "test LLM"
    axes[0].plot(c["coverage student"], c["akurasi pada porsi student"], marker="o", markersize=3.5, color=color, label=label)
    axes[1].plot(c["throughput 1c (dok/s)"], c["cascade macro-F1"], marker="o", markersize=3.5, color=color, label=label)
    teacher_f1 = evaluation.load_results("teacher_xlmr_large_onnx_int8")[split]["macro_f1"]
    axes[1].axhline(teacher_f1, color=color, linestyle=":", linewidth=1.2)
axes[0].invert_xaxis()
axes[0].set_xlabel("porsi artikel yang diputuskan StaticKD (ambang naik →)")
axes[0].set_ylabel("akurasi pada porsi itu")
axes[0].set_title("Akurasi vs coverage StaticKD")
axes[1].set_xscale("log")
axes[1].set_xlabel("throughput cascade, 1 core (dok/s, log)")
axes[1].set_ylabel("macro-F1 cascade")
axes[1].set_title("Cascade: akurasi vs throughput (titik-titik = teacher saja)")
for ax in axes:
    ax.legend(loc="lower left")
fig.tight_layout()
viz.savefig(fig, "fig14_cascade")
plt.show()
cascade_table = cascade[cascade["ambang"].isin([0.0, 0.5, 0.6, 0.7, 0.8, 0.9])]
save_table(cascade_table, "table9_cascade")
cascade_table

**Interpretasi (hasil riset asli, mode REPORT).** Probabilitas StaticKD terkalibrasi cukup baik (ECE dev ≈ 0,05). Dengan ambang 0,8, StaticKD memutuskan ±66% artikel dengan akurasi ±92%, dan cascade mencapai micro-F1 setara teacher. Throughput cascade tetap dibatasi teacher (±1,8 dok/s per core). Karena itu mode cascade cocok untuk jalur batch/near-real-time. Untuk jalur real-time, StaticKD tunggal adalah pilihan yang tepat.

### 8.5 Contoh kualitatif dan analisis kesalahan

Contoh berikut ditulis manual dalam beberapa bahasa, termasuk **bahasa Jawa** yang tidak ada di data teacher maupun data distilasi, untuk menunjukkan transfer lintas bahasa dari embedding yang selaras. Output berupa 3 label teratas beserta probabilitasnya.

In [ ]:
EXAMPLES = {
    "id": "Pemerintah resmi menaikkan harga BBM bersubsidi jenis Pertalite dan Solar mulai Sabtu. Menteri Keuangan menyebut kenaikan ini perlu untuk menjaga APBN, sementara ekonom memperkirakan inflasi akan naik pada kuartal berikutnya.",
    "jv": "Tim nasional Indonesia menang 2-0 nglawan Vietnam ing Stadion Gelora Bung Karno. Pelatih Shin Tae-yong ngandika para pemain main kanthi disiplin lan bakal nyiapake pertandingan sabanjure.",
    "en": "Heavy rain triggered landslides in the mountain villages overnight, killing at least twelve people. Rescue teams are searching for survivors as the national disaster agency warned of more floods.",
    "de": "Die Polizei hat am Montag zwei Verdächtige nach einem bewaffneten Raubüberfall auf eine Bank in Hamburg festgenommen. Die Staatsanwaltschaft ermittelt wegen schweren Raubes.",
    "es": "El nuevo museo de arte contemporáneo abrió sus puertas con una exposición de pintura y fotografía de artistas latinoamericanos, acompañada de conciertos y un festival de cine.",
    "ja": "気象庁は明日、関東地方で大雪になる恐れがあると発表し、交通機関への影響に注意するよう呼びかけた。",
    "ar": "أعلنت وزارة الصحة عن حملة تطعيم جديدة ضد الحصبة في المدارس بعد ارتفاع عدد الإصابات بين الأطفال.",
    "sw": "Walimu kote nchini wameanza mgomo wakidai nyongeza ya mishahara na mazingira bora ya kazi, huku chama chao kikisema mazungumzo na serikali yamekwama.",
}
t0 = time.perf_counter()
probs = student.predict_proba(list(EXAMPLES.values()))
elapsed_ms = (time.perf_counter() - t0) * 1000
rows = []
for (lang, text), p in zip(EXAMPLES.items(), probs):
    top = np.argsort(p)[::-1][:3]
    rows.append({"bahasa": lang, "teks": text[:70] + "...",
                 "top-1": f"{LABELS[top[0]]} ({p[top[0]]:.2f})",
                 "top-2": f"{LABELS[top[1]]} ({p[top[1]]:.2f})",
                 "top-3": f"{LABELS[top[2]]} ({p[top[2]]:.2f})"})
qualitative = pd.DataFrame(rows)
if RUN_TEACHER_DEMO:
    from iptc.teacher import Teacher
    t_logits = Teacher().logits(list(EXAMPLES.values()))
    qualitative["teacher"] = [LABELS[i] for i in t_logits.argmax(1)]
print(f"{len(EXAMPLES)} artikel diklasifikasi StaticKD dalam {elapsed_ms:.1f} ms (CPU)")
qualitative

In [ ]:
# Error analysis: dev documents where the teacher is right and StaticKD is wrong
dev_df = load_emmediatopic("dev")
teacher_dev = evaluation.predictions("teacher_xlmr_large_onnx_int8", "dev")
student_dev = PROBS["dev"].argmax(1)
mask = (teacher_dev == Y_TRUE["dev"]) & (student_dev != Y_TRUE["dev"])
print(f"Teacher benar & StaticKD salah: {mask.sum()} dok ({mask.mean() * 100:.1f}%) | "
      f"StaticKD benar & teacher salah: {((student_dev == Y_TRUE['dev']) & (teacher_dev != Y_TRUE['dev'])).sum()} dok")
errors = dev_df[mask].assign(prediksi_student=[LABELS[i] for i in student_dev[mask]],
                             confidence=PROBS["dev"].max(1)[mask]).sample(min(6, mask.sum()), random_state=1)
errors["teks"] = errors["text"].str.replace("\n", " ").str[:160] + "..."
errors[["lang", "label", "prediksi_student", "confidence", "teks"]]

---
## 9. Ringkasan temuan dan ekspor untuk paper

Sel berikut merangkum angka-angka kunci langsung dari hasil (bukan diketik manual), sehingga teks paper selalu konsisten dengan hasil eksperimen. Semua tabel ada di `results/paper_tables/` (CSV + LaTeX) dan gambar di `results/figures/` (PNG 300 dpi + PDF).

In [ ]:
def get(key, split, metric="macro_f1"):
    return evaluation.load_results(key)[split][metric]

findings = [
    f"Teacher (ONNX int8) — dev macro-F1 {get('teacher_xlmr_large_onnx_int8', 'dev'):.3f}, "
    f"test LLM 16 bahasa {get('teacher_xlmr_large_onnx_int8', 'llm_test'):.3f}; "
    f"di CPU {ref.loc['teacher_xlmr_large_onnx_int8', 'Latensi p50 1c (ms)']:.0f} ms/artikel, "
    f"{ref.loc['teacher_xlmr_large_onnx_int8', 'Throughput 1c (dok/s)']:.2f} dok/s per core.",
    f"StaticKD — dev macro-F1 {get('static_final_table', 'dev'):.3f} "
    f"({get('static_final_table', 'dev') / get('teacher_xlmr_large_onnx_int8', 'dev') * 100:.0f}% teacher), "
    f"test LLM {get('static_final_table', 'llm_test'):.3f} "
    f"({get('static_final_table', 'llm_test') / get('teacher_xlmr_large_onnx_int8', 'llm_test') * 100:.0f}% teacher); "
    f"{ref.loc['static_final_table', 'Latensi p50 1c (ms)']:.1f} ms/artikel, "
    f"{ref.loc['static_final_table', 'Throughput 1c (dok/s)']:.0f} dok/s per core, file {ref.loc['static_final_table', 'Ukuran file (MB)']:.0f} MB.",
    f"Percepatan throughput 1 core: {speedup:.0f}x vs teacher ONNX int8, {speedup_pt:.0f}x vs teacher PyTorch.",
    f"fastText + KD (quantized) — test LLM {get('ft_teacher_subword_q', 'llm_test'):.3f}; "
    f"fastText setting Kuzman (label GPT-4o, 4 bahasa) — test LLM {get('ft_gpt_subword', 'llm_test'):.3f}.",
    f"Faktor per bahasa: Spearman kesepakatan vs log(data latih) ρ = {rho_data:.2f} (p = {p_data:.1e}); "
    f"R² data = {r2_data:.2f}, aksara saja = {r2_script:.2f}, data + aksara = {r2_both:.2f}.",
    f"Kalibrasi StaticKD: ECE dev = {calib_rows[0]['ECE']:.3f}, test LLM = {calib_rows[1]['ECE']:.3f}.",
]
c8 = cascade.query("set == 'dev' and ambang == 0.8").iloc[0]
findings.append(f"Cascade (ambang 0,8): StaticKD memutuskan {c8['coverage student'] * 100:.0f}% artikel, "
                f"micro-F1 dev {c8['cascade micro-F1']:.3f} vs teacher {get('teacher_xlmr_large_onnx_int8', 'dev', 'micro_f1'):.3f}, "
                f"throughput {c8['throughput 1c (dok/s)']:.1f} dok/s per core.")
for f in findings:
    print("•", f)
(RESULTS_DIR / "key_findings.txt").write_text("\n".join(findings), encoding="utf8")

print("\nTabel:", sorted(p.name for p in TABLES_DIR.glob("*.csv")))
print("Gambar:", sorted(p.name for p in config.FIGURES_DIR.glob("*.png")))

In [ ]:
%%writefile iptc/bundle.py
"""Pack the artefacts needed by REPORT mode into one zip (to share results or re-run analyses)."""
import zipfile

from iptc.config import BASE_DIR, MODELS_DIR, PROCESSED_DIR, RESULTS_DIR

BUNDLE_FILES = [
    *sorted((RESULTS_DIR / "accuracy").glob("*.json")),
    *sorted(RESULTS_DIR.glob("cpu_benchmark*.jsonl")),
    RESULTS_DIR / "training_history.json",
    RESULTS_DIR / "data_stats.json",
    PROCESSED_DIR / "llm_test.jsonl",
    PROCESSED_DIR / "emm_dev.jsonl",
]
BUNDLE_DIRS = [MODELS_DIR / "static" / "static_final_table", MODELS_DIR / "static" / "static_final_table_compact10"]


def make_report_bundle(name: str = "iptc_statickd_bundle.zip") -> str:
    out = BASE_DIR / name
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in BUNDLE_FILES:
            if f.exists():
                zf.write(f, f.relative_to(BASE_DIR))
        for d in BUNDLE_DIRS:
            for f in sorted(d.glob("*")):
                zf.write(f, f.relative_to(BASE_DIR))
    return str(out)

In [ ]:
if MODE == "FULL":
    from iptc.bundle import make_report_bundle
    print("Bundle REPORT dibuat:", make_report_bundle())

---
## 10. Keterbatasan dan catatan reproduktibilitas

**Ancaman validitas**
1. **Test set manusia tidak diakses.** Dev berlabel GPT-4o dan test set 16 bahasa berlabel LLM (satu anotator, 20 dokumen per bahasa, sehingga CI lebar). Langkah validasi berikutnya: meminta IPTC-test (1.129 dokumen berlabel manusia) ke penulis dan melaporkan skor StaticKD di sana. Repositori benchmark mereka menerima submisi.
2. **Anotator test set LLM** adalah Claude, dari keluarga model yang sama dengan asisten yang membantu menjalankan riset. Anotasi dibuat buta terhadap prediksi model dengan prompt Kuzman, dan kesepakatannya dengan teacher setara dengan kesepakatan manusia–manusia (Bagian 3.4). Validasi manusia pada sampel tetap disarankan.
3. **Batas atas student adalah teacher.** Kesalahan dan bias teacher (label *lifestyle*, *human interest*, *society*) ikut tertiru, bahkan lebih tajam.
4. **Kesepakatan 70 bahasa** mengukur *fidelity* terhadap teacher, bukan kebenaran absolut.
5. **Benchmark CPU** diukur di laptop dengan CPU hybrid (dipin ke P-core). Angka absolut di server berbeda, tetapi rasio antar model tetap informatif. Mode FULL mengulang benchmark di CPU Colab.

**Reproduktibilitas**
- Seed tetap (`seed=0` untuk training, `13/21/7` untuk sampling data). Hasil CC-News bergantung pada isi arsip CloverSearch pada saat unduh.
- Versi paket yang dipakai pada riset asli: torch 2.6, transformers 5.17, tokenizers 0.23, onnxruntime 1.30, scikit-learn 1.9, fasttext-wheel 0.9.2, numpy 2.5.
- Semua artefak antara (kandidat CC-News, logit teacher, cache token, model) disimpan di Drive sehingga setiap tahap dapat dijalankan ulang secara terpisah.

**Referensi utama**
- T. Kuzman & N. Ljubešić (2025). *LLM Teacher-Student Framework for Text Classification With No Manually Annotated Data: A Case Study in IPTC News Topic Classification.* IEEE Access 13.
- G. Hinton, O. Vinyals, J. Dean (2015). *Distilling the Knowledge in a Neural Network.* NIPS DL Workshop.
- A. Joulin, E. Grave, P. Bojanowski, T. Mikolov (2017). *Bag of Tricks for Efficient Text Classification.* EACL.
- S. Tulkens & T. van Dongen (2024). *Model2Vec: Fast State-of-the-Art Static Embeddings.* (potion-multilingual-128M).
- W. Wang dkk. (2021). *MiniLMv2: Multi-Head Self-Attention Relation Distillation for Compressing Pretrained Transformers.* ACL Findings.
- A. Conneau dkk. (2020). *Unsupervised Cross-lingual Representation Learning at Scale (XLM-R).* ACL.